In [9]:
from pydantic import BaseModel
from google.genai import types

class MetricChoice(BaseModel):
    metric: str
    weight: float
    explanation: str

class RankingPlan(BaseModel):
    choices: list[MetricChoice]
    unsupported: list[str]
    cautions: list[str]

def preference_agent(student_request):
    instructions = """
    You are the Preference Interpreter for RankMyCollege.

    Translate a student's college preferences into
    measurable ranking criteria.

    Rules:
    - Only use metrics from available_metrics.
    - Choose up to 6 relevant metrics.
    - Assign positive importance weights.
    - Explain each metric selection.
    - Identify preferences not supported by the data.
    - Never invent college statistics.
    - Do not assume more selective means better.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=json.dumps({
            "student_request": student_request,
            "available_metrics": rankable_metrics
        }),
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            response_mime_type="application/json",
            response_schema=RankingPlan,
            temperature=0.1
        )
    )

    return RankingPlan.model_validate_json(response.text)

In [12]:
student_request = """
I want an affordable college with strong
career earnings, low student debt, and
high graduation rates. Prestige doesn't
matter to me.
"""

result = preference_agent(student_request)

print(result.model_dump_json(indent=2))

NameError: name 'rankable_metrics' is not defined

In [11]:
import json

In [13]:
import json
import pandas as pd
from pathlib import Path

# Locate your uploaded files
def find_file(filename):
    for folder in [Path.cwd(), Path("/content")]:
        path = folder / filename
        if path.exists():
            return path
    raise FileNotFoundError(f"Upload {filename} first!")

# Load metric catalog
with open(find_file("metrics.json"), encoding="utf-8") as f:
    metrics = json.load(f)

# Load college database
colleges = pd.read_csv(
    find_file("colleges_plus_scorecard.csv")
)

# Create the variable Agent 1 needs
rankable_metrics = {
    name: info
    for name, info in metrics.items()
    if info.get("direction") in ("higher", "lower")
    and name in colleges.columns
}

print("✅ Data loaded successfully!")
print("Colleges:", len(colleges))
print("Total metrics:", len(metrics))
print("Rankable metrics:", len(rankable_metrics))
print("Available:", list(rankable_metrics))

✅ Data loaded successfully!
Colleges: 2796
Total metrics: 11
Rankable metrics: 9
Available: ['graduation_rate_150pct', 'bachelors_4yr_completion_rate', 'retention_rate', 'student_faculty_ratio', 'average_net_price', 'MD_EARN_WNE_P10', 'MD_EARN_WNE_P6', 'DEBT_MDN', 'GRAD_DEBT_MDN']


In [14]:
result = preference_agent(student_request)

print(result.model_dump_json(indent=2))

{
  "choices": [
    {
      "metric": "average_net_price",
      "weight": 5.0,
      "explanation": "Addresses the desire for an affordable college by measuring the average net price after grant aid."
    },
    {
      "metric": "DEBT_MDN",
      "weight": 4.0,
      "explanation": "Directly tracks low student debt using the median federal student loan debt among borrowers."
    },
    {
      "metric": "MD_EARN_WNE_P10",
      "weight": 4.0,
      "explanation": "Captures strong career earnings by evaluating median earnings 10 years after entering college."
    },
    {
      "metric": "graduation_rate_150pct",
      "weight": 3.0,
      "explanation": "Measures high graduation rates based on completion within 150% of normal time."
    }
  ],
  "unsupported": [
    "Specific college rankings or prestige metrics, as the data does not measure prestige."
  ],
  "cautions": [
    "Do not assume more selective colleges are better or more affordable."
  ]
}


In [15]:
def auditor_agent(student_request, first_plan):
    instructions = """
    You are RankMyCollege's Preference Auditor.

    Your job is to independently review another AI
    agent's interpretation of a student's preferences.

    Review the original student request and compare
    it with the proposed ranking metrics and weights.

    YOUR RESPONSIBILITIES:

    1. Check that the selected metrics actually
       represent the student's preferences.

    2. Identify irrelevant, duplicated, or
       unsupported metrics.

    3. Check whether the relative weights make sense.

    4. Correct mistakes in the first agent's plan.

    5. Remove irrelevant unsupported claims.
       If a student says something doesn't matter,
       it is not an unsupported requirement.

    6. Explain data limitations clearly.

    IMPORTANT RULES:
    - Only use available_metrics.
    - Never invent university statistics.
    - Do not change correct decisions unnecessarily.
    - Do not assume prestige equals educational quality.
    - Average net price is not a personalized aid offer.
    - Earnings 10 years after entry are not necessarily
      earnings 10 years after graduation.
    - Return a corrected RankingPlan in JSON format.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=json.dumps({
            "student_request": student_request,
            "available_metrics": rankable_metrics,
            "first_agent_plan": first_plan.model_dump()
        }),
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            response_mime_type="application/json",
            response_schema=RankingPlan,
            temperature=0.1
        )
    )

    return RankingPlan.model_validate_json(
        response.text
    )

print("✅ Agent 2 created!")

✅ Agent 2 created!


In [16]:
# Agent 1 has already produced its result
print("========== AGENT 1 ==========")
print(result.model_dump_json(indent=2))

# Agent 2 independently reviews Agent 1
audited_result = auditor_agent(
    student_request,
    result
)

print("\n========== AGENT 2 ==========")
print(audited_result.model_dump_json(indent=2))

========== AGENT 1 ==========
{
  "choices": [
    {
      "metric": "average_net_price",
      "weight": 5.0,
      "explanation": "Addresses the desire for an affordable college by measuring the average net price after grant aid."
    },
    {
      "metric": "DEBT_MDN",
      "weight": 4.0,
      "explanation": "Directly tracks low student debt using the median federal student loan debt among borrowers."
    },
    {
      "metric": "MD_EARN_WNE_P10",
      "weight": 4.0,
      "explanation": "Captures strong career earnings by evaluating median earnings 10 years after entering college."
    },
    {
      "metric": "graduation_rate_150pct",
      "weight": 3.0,
      "explanation": "Measures high graduation rates based on completion within 150% of normal time."
    }
  ],
  "unsupported": [
    "Specific college rankings or prestige metrics, as the data does not measure prestige."
  ],
  "cautions": [
    "Do not assume more selective colleges are better or more affordable."
  ]
}


In [17]:
print("ORIGINAL WEIGHTS:")
for choice in result.choices:
    print(choice.metric, ":", choice.weight)

print("\nAUDITED WEIGHTS:")
for choice in audited_result.choices:
    print(choice.metric, ":", choice.weight)

print("\nORIGINAL UNSUPPORTED:")
print(result.unsupported)

print("\nAUDITED UNSUPPORTED:")
print(audited_result.unsupported)

print("\nAUDITOR CAUTIONS:")
for caution in audited_result.cautions:
    print("-", caution)

ORIGINAL WEIGHTS:
average_net_price : 5.0
DEBT_MDN : 4.0
MD_EARN_WNE_P10 : 4.0
graduation_rate_150pct : 3.0

AUDITED WEIGHTS:
average_net_price : 5.0
DEBT_MDN : 4.0
MD_EARN_WNE_P10 : 4.0
graduation_rate_150pct : 3.0

ORIGINAL UNSUPPORTED:
['Specific college rankings or prestige metrics, as the data does not measure prestige.']

AUDITED UNSUPPORTED:
[]

AUDITOR CAUTIONS:
- Average net price is not a personalized aid offer and may vary significantly from your actual cost.
- Earnings 10 years after entry are not necessarily earnings 10 years after graduation.


In [19]:
import math

def validate_weights(plan):
    weights = {}

    for choice in plan.choices:
        metric = choice.metric
        weight = choice.weight

        if metric not in rankable_metrics:
            raise ValueError(f"Unknown metric: {metric}")

        if metric in weights:
            raise ValueError(f"Duplicate metric: {metric}")

        if not math.isfinite(weight) or weight <= 0:
            raise ValueError(f"Invalid weight: {weight}")

        weights[metric] = weight

    if not weights:
        raise ValueError("No valid metrics selected.")

    total = sum(weights.values())

    return {
        metric: weight / total
        for metric, weight in weights.items()
    }


# Validate Agent 2's final decisions
weights = validate_weights(audited_result)

print("✅ Ranking weights approved!\n")

for metric, weight in weights.items():
    print(f"{metric}: {weight:.2%}")

print("\nTotal:", sum(weights.values()))

✅ Ranking weights approved!

average_net_price: 31.25%
DEBT_MDN: 25.00%
MD_EARN_WNE_P10: 25.00%
graduation_rate_150pct: 18.75%

Total: 1.0


In [20]:

import pandas as pd
import math

def rank_colleges(colleges, weights, metric_catalog):
    """
    Rank colleges using user-selected metric weights.
    Works with AI-generated or manually edited weights.
    """

    # Validate the weights
    selected = {}

    for metric, weight in weights.items():
        if metric not in metric_catalog:
            raise ValueError(f"Unknown metric: {metric}")

        if metric not in colleges.columns:
            raise ValueError(f"Missing column: {metric}")

        if not isinstance(weight, (int, float)):
            raise ValueError(f"Non-numeric weight: {metric}")

        if not math.isfinite(weight) or weight < 0:
            raise ValueError(f"Invalid weight: {metric}")

        if weight > 0:
            selected[metric] = weight

    if not selected:
        raise ValueError("Select at least one metric.")

    # Normalize weights to total 100%
    total = sum(selected.values())
    normalized_weights = {
        metric: value / total
        for metric, value in selected.items()
    }

    df = colleges.copy()

    # Only include colleges with undergraduates
    df["undergraduate_enrollment"] = pd.to_numeric(
        df["undergraduate_enrollment"],
        errors="coerce"
    )

    df = df[df["undergraduate_enrollment"] > 0].copy()

    for metric in selected:
        df[metric] = pd.to_numeric(
            df[metric], errors="coerce"
        )

    # Clean invalid net prices
    if "average_net_price" in selected:
        df.loc[
            df["average_net_price"] < 0,
            "average_net_price"
        ] = float("nan")

    # Prototype: rank only complete records
    df = df.dropna(subset=list(selected)).copy()

    if df.empty:
        raise ValueError("No eligible colleges found.")

    # Calculate weighted percentile scores
    df["final_score"] = 0.0

    for metric, weight in normalized_weights.items():
        direction = metric_catalog[metric]["direction"]

        if direction not in ("higher", "lower"):
            raise ValueError(
                f"Unsupported direction for {metric}"
            )

        percentile = df[metric].rank(
            pct=True,
            ascending=(direction == "higher")
        ) * 100

        contribution = percentile * weight

        df[f"{metric}_contribution"] = contribution
        df["final_score"] += contribution

    return df.sort_values(
        "final_score", ascending=False
    ).reset_index(drop=True)


print("✅ Dynamic ranking engine created!")


✅ Dynamic ranking engine created!


In [21]:

ranked = rank_colleges(
    colleges,
    weights,
    rankable_metrics
)

print("Colleges ranked:", len(ranked))

display(
    ranked[
        ["college_name", "state", "final_score"]
    ].head(10)
)


Colleges ranked: 1988


,college_name,state,final_score
0,United States Merchant Marine Academy,NY,96.664361
1,CUNY Bernard M Baruch College,NY,90.335765
2,California Institute of Technology,CA,88.388141
3,University of North Carolina at Chapel Hill,NC,86.952968
4,Williams College,MA,86.259746
5,CUNY Hunter College,NY,84.348277
6,CUNY City College,NY,84.172221
7,Princeton University,NJ,84.055898
8,University of Washington-Seattle Campus,WA,83.994593
9,University of Florida-Online,FL,82.774774


In [22]:

custom_weights = {
    "average_net_price": 1,
    "DEBT_MDN": 1,
    "MD_EARN_WNE_P10": 8,
    "graduation_rate_150pct": 3
}

custom_rankings = rank_colleges(
    colleges,
    custom_weights,
    rankable_metrics
)

print("🏆 TOP 10 WITH CUSTOM WEIGHTS")

display(
    custom_rankings[
        ["college_name", "state", "final_score"]
    ].head(10)
)


🏆 TOP 10 WITH CUSTOM WEIGHTS


,college_name,state,final_score
0,California Institute of Technology,CA,96.676211
1,United States Merchant Marine Academy,NY,96.355053
2,Princeton University,NJ,95.474772
3,Williams College,MA,93.969587
4,University of California-Berkeley,CA,92.975159
5,Massachusetts Institute of Technology,MA,92.961616
6,Harvard University,MA,92.938400
7,Yale University,CT,92.926792
8,Stanford University,CA,92.890033
9,Dartmouth College,NH,92.385080


In [23]:

import pandas as pd
import math

# Start with our existing college database
comparison_pool = colleges.copy()

# Only include colleges with undergraduate students
comparison_pool["undergraduate_enrollment"] = pd.to_numeric(
    comparison_pool["undergraduate_enrollment"],
    errors="coerce"
)

comparison_pool = comparison_pool[
    comparison_pool["undergraduate_enrollment"] > 0
].copy()

# Convert all rankable metrics to numbers
for metric in rankable_metrics:
    comparison_pool[metric] = pd.to_numeric(
        comparison_pool[metric],
        errors="coerce"
    )

# Negative net prices are invalid
if "average_net_price" in comparison_pool.columns:
    comparison_pool.loc[
        comparison_pool["average_net_price"] < 0,
        "average_net_price"
    ] = float("nan")

# Fixed comparison group
comparison_pool = comparison_pool.dropna(
    subset=list(rankable_metrics)
).copy()

print("Original colleges:", len(colleges))
print("Colleges in fixed comparison pool:", len(comparison_pool))
print("Metrics supported:", len(rankable_metrics))


Original colleges: 2796
Colleges in fixed comparison pool: 1709
Metrics supported: 9


In [24]:

def rank_colleges(pool, selected_weights, catalog):
    active = {}

    for metric, weight in selected_weights.items():
        if metric not in catalog:
            raise ValueError(f"Unknown metric: {metric}")

        if catalog[metric]["direction"] not in ("higher", "lower"):
            raise ValueError(f"Unsupported direction: {metric}")

        if isinstance(weight, bool) or not isinstance(weight, (int, float)):
            raise ValueError(f"Invalid weight type: {metric}")

        if not math.isfinite(weight) or weight < 0:
            raise ValueError(f"Invalid weight: {metric}")

        if weight > 0:
            active[metric] = weight

    if not active:
        raise ValueError("At least one metric must have a positive weight.")

    total = sum(active.values())
    normalized = {m: w / total for m, w in active.items()}

    df = pool.copy()
    df["final_score"] = 0.0

    for metric, weight in normalized.items():
        # Higher metrics: larger values receive higher scores.
        # Lower metrics: smaller values receive higher scores.
        ascending = catalog[metric]["direction"] == "higher"

        percentile = df[metric].rank(
            pct=True,
            ascending=ascending
        ) * 100

        contribution = percentile * weight
        df[f"{metric}_contribution"] = contribution
        df["final_score"] += contribution

    df = df.sort_values(
        ["final_score", "college_name"],
        ascending=[False, True]
    ).reset_index(drop=True)

    df["rank"] = df.index + 1
    return df


print("✅ Ranking engine ready!")


✅ Ranking engine ready!


In [25]:

# Agent 2's approved weights
ai_rankings = rank_colleges(
    comparison_pool,
    weights,
    rankable_metrics
)

# Simulate a user prioritizing earnings
custom_weights = {
    "average_net_price": 1,
    "DEBT_MDN": 1,
    "MD_EARN_WNE_P10": 8,
    "graduation_rate_150pct": 3
}

custom_rankings = rank_colleges(
    comparison_pool,
    custom_weights,
    rankable_metrics
)

print("🏆 TOP 10 — AI PREFERENCES")
display(
    ai_rankings[
        ["rank", "college_name", "state", "final_score"]
    ].head(10)
)

print("\n🏆 TOP 10 — CUSTOM PREFERENCES")
display(
    custom_rankings[
        ["rank", "college_name", "state", "final_score"]
    ].head(10)
)

# Verify the same colleges were compared
assert set(ai_rankings["unitid"]) == set(custom_rankings["unitid"])
assert ai_rankings["final_score"].between(0, 100).all()
assert custom_rankings["final_score"].between(0, 100).all()

print("\n✅ Same colleges in both comparisons!")
print("✅ All ranking scores are valid!")


🏆 TOP 10 — AI PREFERENCES


,rank,college_name,state,final_score
0,1,United States Merchant Marine Academy,NY,97.182197
1,2,CUNY Bernard M Baruch College,NY,91.910474
2,3,California Institute of Technology,CA,91.659962
3,4,University of North Carolina at Chapel Hill,NC,89.931978
4,5,Williams College,MA,88.748903
5,6,Princeton University,NJ,87.823654
6,7,University of Washington-Seattle Campus,WA,87.137946
7,8,University of California-San Diego,CA,86.006071
8,9,University of Florida-Online,FL,85.605617
9,10,University of Florida,FL,85.174078



🏆 TOP 10 — CUSTOM PREFERENCES


,rank,college_name,state,final_score
0,1,California Institute of Technology,CA,97.585183
1,2,Princeton University,NJ,96.540937
2,3,United States Merchant Marine Academy,NY,96.212360
3,4,Williams College,MA,94.292659
4,5,Massachusetts Institute of Technology,MA,93.869559
5,6,Harvard University,MA,93.806545
6,7,Yale University,CT,93.730027
7,8,University of California-Berkeley,CA,93.727776
8,9,Stanford University,CA,93.459963
9,10,Dartmouth College,NH,93.153891



✅ Same colleges in both comparisons!
✅ All ranking scores are valid!


In [26]:

import json

def prepare_explainer_evidence(
    ai_rankings,
    custom_rankings,
    ai_weights,
    custom_weights
):
    # Compare top 3 colleges from both scenarios
    college_ids = list(dict.fromkeys(
        ai_rankings["unitid"].head(3).tolist()
        + custom_rankings["unitid"].head(3).tolist()
    ))

    ai_lookup = ai_rankings.set_index("unitid")
    custom_lookup = custom_rankings.set_index("unitid")

    selected_metrics = list(dict.fromkeys(
        list(ai_weights) + list(custom_weights)
    ))

    def normalize(w):
        total = sum(w.values())
        return {k: v / total for k, v in w.items()}

    evidence = {
        "student_request": student_request,
        "ai_weights": normalize(ai_weights),
        "custom_weights": normalize(custom_weights),
        "colleges": []
    }

    for unitid in college_ids:
        original = ai_lookup.loc[unitid]
        changed = custom_lookup.loc[unitid]

        metrics_data = {}

        for metric in selected_metrics:
            metrics_data[metric] = {
                "actual_value": float(original[metric]),
                "direction": rankable_metrics[metric]["direction"],
                "ai_contribution": float(
                    original.get(f"{metric}_contribution", 0)
                ),
                "custom_contribution": float(
                    changed.get(f"{metric}_contribution", 0)
                )
            }

        evidence["colleges"].append({
            "college_name": str(original["college_name"]),
            "ai_rank": int(original["rank"]),
            "custom_rank": int(changed["rank"]),
            "ai_score": float(original["final_score"]),
            "custom_score": float(changed["final_score"]),
            "metrics": metrics_data
        })

    return evidence


evidence = prepare_explainer_evidence(
    ai_rankings,
    custom_rankings,
    weights,
    custom_weights
)

print("Evidence prepared for:",
      len(evidence["colleges"]), "colleges")
print(json.dumps(evidence, indent=2)[:3000])


Evidence prepared for: 4 colleges
{
  "student_request": "\nI want an affordable college with strong\ncareer earnings, low student debt, and\nhigh graduation rates. Prestige doesn't\nmatter to me.\n",
  "ai_weights": {
    "average_net_price": 0.3125,
    "DEBT_MDN": 0.25,
    "MD_EARN_WNE_P10": 0.25,
    "graduation_rate_150pct": 0.1875
  },
  "custom_weights": {
    "average_net_price": 0.07692307692307693,
    "DEBT_MDN": 0.07692307692307693,
    "MD_EARN_WNE_P10": 0.6153846153846154,
    "graduation_rate_150pct": 0.23076923076923078
  },
  "colleges": [
    {
      "college_name": "United States Merchant Marine Academy",
      "ai_rank": 1,
      "custom_rank": 3,
      "ai_score": 97.18219719133995,
      "custom_score": 96.21235990457757,
      "metrics": {
        "average_net_price": {
          "actual_value": 4101.0,
          "direction": "lower",
          "ai_contribution": 31.067144528964306,
          "custom_contribution": 7.647297114821984
        },
        "DEBT_MDN"

In [43]:

from pydantic import BaseModel
from google.genai import types

class ExplanationReport(BaseModel):
    original_winner_explanation: str
    custom_winner_explanation: str
    why_rankings_changed: str
    key_tradeoffs: list[str]



def explainer_agent(evidence):
    instructions = """
    You are RankMyCollege's college match explainer.

    AUDIENCE:
    High school seniors, ages 17-18.

    YOUR PERSONALITY:
    Friendly, clear, concise, and relatable.
    Sound like a knowledgeable older student,
    not a university marketing department.

    STRICT WRITING RULES:
    - Original winner: maximum 25 words.
    - Custom winner: maximum 25 words.
    - Ranking changes: maximum 30 words.
    - Give exactly 2 tradeoffs.
    - Each tradeoff: maximum 18 words.
    - Use simple, conversational English.
    - Speak directly to the student.
    - Use "you" and "your" where natural.
    - Avoid long explanations.
    - Avoid promotional exaggeration.
    - Never use phrases like:
      "incredible value", "fantastic balance",
      "exceptionally high", "soared past",
      "post-entry earnings", or "institution".
    - No filler or marketing language.
    - No separate data limitations section.

    FACTUAL RULES:
    - Use only the supplied evidence.
    - Never invent statistics or comparisons.
    - Never recalculate or change rankings.
    - Never promise admission or employment.
    - Do not confuse earnings after college entry
      with earnings after graduation.
    - Do not assume average net price equals
      the student's personal cost.

    PERSONALIZATION RULES:
    - Always name the specific college being explained.
    - Never start with "This school" or "This choice".
    - Identify the original #1 college using ai_rank = 1.
    - Identify the custom #1 college using custom_rank = 1.
    - Explain one concrete reason each college ranked first.
    - Refer to actual metrics and score contributions.
    - Keep winner explanations under 35 words.
    - Keep ranking-change explanations under 30 words.
    - Give exactly 2 short tradeoffs.
    - Name the colleges involved in each tradeoff.
    - Use "average net price", not "your actual cost".
    - Use "reported graduate or former-student earnings"
      only when supported by the supplied metric definition.
    - Never predict a student's future salary or debt.
    - Never invent numbers or statistics.
    - Don't copy generic example sentences.
    - Write directly to the student using friendly language.
    When referencing MD_EARN_WNE_P10, say
    "reported earnings 10 years after entering college."

    Never call this metric graduate earnings.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=json.dumps(evidence, allow_nan=False),
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            response_mime_type="application/json",
            response_schema=ExplanationReport,
            temperature=0.1
        )
    )

    return ExplanationReport.model_validate_json(
        response.text
    )

print("✅ Agent 3 updated for high school seniors!")



✅ Agent 3 updated for high school seniors!


In [35]:

explanation = explainer_agent(evidence)

print("🏆 WHY THE ORIGINAL #1 COLLEGE WON")
print(explanation.original_winner_explanation)

print("\n🏆 WHY THE CUSTOM #1 COLLEGE WON")
print(explanation.custom_winner_explanation)

print("\n🔄 WHY THE RANKINGS CHANGED")
print(explanation.why_rankings_changed)

print("\n⚖️ IMPORTANT TRADEOFFS")
for item in explanation.key_tradeoffs:
    print("-", item)

🏆 WHY THE ORIGINAL #1 COLLEGE WON
This school fits your budget and debt priorities best. Its low price and solid scores helped it rank first.

🏆 WHY THE CUSTOM #1 COLLEGE WON
You made career earnings your biggest priority. That is why this school moved to the top.

🔄 WHY THE RANKINGS CHANGED
When you put more weight on earnings, schools with the highest career earnings scores moved up.

⚖️ IMPORTANT TRADEOFFS
- Lower net price often means higher student debt or lower career earnings.
- Higher graduation rates usually come with a higher net price tag.


In [38]:

explanation = explainer_agent(evidence)

sections = {
    "🏆 YOUR TOP MATCH":
        explanation.original_winner_explanation,

    "🎯 YOUR NEW TOP MATCH":
        explanation.custom_winner_explanation,

    "🔄 WHAT CHANGED?":
        explanation.why_rankings_changed
}

for heading, text in sections.items():
    print(f"\n{heading}")
    print(text)
    print(f"({len(text.split())} words)")

print("\n⚖️ THE TRADEOFFS")
for tradeoff in explanation.key_tradeoffs:
    print("-", tradeoff)
    print(f"({len(tradeoff.split())} words)")



🏆 YOUR TOP MATCH
United States Merchant Marine Academy ranked first because it balanced a low average net price with high reported former-student earnings.
(20 words)

🎯 YOUR NEW TOP MATCH
California Institute of Technology took the top spot because you weighted reported former-student earnings much more heavily.
(17 words)

🔄 WHAT CHANGED?
When you heavily prioritized career earnings over low net price and student debt, schools with very high reported salaries jumped up.
(21 words)

⚖️ THE TRADEOFFS
- California Institute of Technology has higher reported earnings than United States Merchant Marine Academy, but also a higher average net price.
(21 words)
- United States Merchant Marine Academy offers a lower average net price than Princeton University, but a lower graduation rate.
(19 words)


In [39]:

def run_rankmycollege(student_request, edited_weights=None):

    print("🧠 Agent 1: Understanding the student...")
    initial_plan = preference_agent(student_request)

    print("🛡️ Agent 2: Checking preferences...")
    approved_plan = auditor_agent(
        student_request,
        initial_plan
    )

    # Validate Agent 2's metric weights
    ai_weights = validate_weights(approved_plan)

    # Use manually edited weights if provided
    selected_weights = (
        edited_weights if edited_weights is not None
        else ai_weights
    )

    print("📊 Ranking colleges...")

    ai_rankings = rank_colleges(
        comparison_pool,
        ai_weights,
        rankable_metrics
    )

    custom_rankings = rank_colleges(
        comparison_pool,
        selected_weights,
        rankable_metrics
    )

    # Normalize the edited weights
    active = {
        k: v for k, v in selected_weights.items()
        if v > 0
    }

    if not active:
        raise ValueError("At least one weight must be positive.")

    total = sum(active.values())

    normalized_custom = {
        k: v / total for k, v in active.items()
    }

    # Prepare verified results for Agent 3
    evidence = prepare_explainer_evidence(
        ai_rankings,
        custom_rankings,
        ai_weights,
        normalized_custom
    )

    evidence["student_request"] = student_request

    print("💬 Agent 3: Explaining results...")
    explanation = explainer_agent(evidence)

    print("✅ Full backend pipeline completed!")

    return {
        "initial_plan": initial_plan,
        "approved_plan": approved_plan,
        "ai_weights": ai_weights,
        "ai_rankings": ai_rankings,
        "custom_rankings": custom_rankings,
        "explanation": explanation
    }


In [40]:

test_request = """
I want an affordable university with low student
debt, strong career earnings, and high graduation
rates. Prestige isn't important to me.
"""

edited_weights = {
    "average_net_price": 1,
    "DEBT_MDN": 1,
    "MD_EARN_WNE_P10": 8,
    "graduation_rate_150pct": 3
}

backend_result = run_rankmycollege(
    test_request,
    edited_weights
)


🧠 Agent 1: Understanding the student...
🛡️ Agent 2: Checking preferences...
📊 Ranking colleges...
💬 Agent 3: Explaining results...
✅ Full backend pipeline completed!


In [44]:

print("🏆 TOP 5 ORIGINAL MATCHES")
display(
    backend_result["ai_rankings"][
        ["rank", "college_name", "final_score"]
    ].head()
)

print("\n🎯 TOP 5 AFTER ADJUSTING PRIORITIES")
display(
    backend_result["custom_rankings"][
        ["rank", "college_name", "final_score"]
    ].head()
)

report = backend_result["explanation"]

print("\n💬 YOUR TOP MATCH")
print(report.original_winner_explanation)

print("\n🎯 YOUR NEW TOP MATCH")
print(report.custom_winner_explanation)

print("\n🔄 WHAT CHANGED?")
print(report.why_rankings_changed)

print("\n⚖️ THE TRADEOFFS")
for item in report.key_tradeoffs:
    print("-", item)


🏆 TOP 5 ORIGINAL MATCHES


,rank,college_name,final_score
0,1,United States Merchant Marine Academy,96.934530
1,2,California Institute of Technology,92.298940
2,3,CUNY Bernard M Baruch College,91.299330
3,4,University of North Carolina at Chapel Hill,89.914505
4,5,Williams College,89.771471



🎯 TOP 5 AFTER ADJUSTING PRIORITIES


,rank,college_name,final_score
0,1,California Institute of Technology,97.585183
1,2,Princeton University,96.540937
2,3,United States Merchant Marine Academy,96.212360
3,4,Williams College,94.292659
4,5,Massachusetts Institute of Technology,93.869559



💬 YOUR TOP MATCH
United States Merchant Marine Academy took the top spot thanks to its low average net price and low student debt.

🎯 YOUR NEW TOP MATCH
California Institute of Technology won your custom search because of its very high reported graduate earnings.

🔄 WHAT CHANGED?
When you put more weight on career earnings, California Institute of Technology climbed past United States Merchant Marine Academy.

⚖️ THE TRADEOFFS
- California Institute of Technology has higher reported graduate earnings than United States Merchant Marine Academy, but a higher average net price.
- United States Merchant Marine Academy offers a lower average net price than California Institute of Technology, but lower reported graduate earnings.


In [45]:

print("📚 AVAILABLE DATASET COLUMNS:")
print(colleges.columns.tolist())

print("\n🏫 INSTITUTION TYPES:")
print(
    colleges["institution_type"].value_counts(
        dropna=False
    )
)

print("\n🏛️ COLLEGE SECTORS:")
print(
    colleges["sector"].value_counts(
        dropna=False
    )
)

print("\n📊 COMPARISON POOL:")
print("Original colleges:", len(colleges))
print("Currently rankable:", len(comparison_pool))
print("Excluded from pool:",
      len(colleges) - len(comparison_pool))


📚 AVAILABLE DATASET COLUMNS:
['unitid', 'college_name', 'city', 'state', 'sector', 'institution_type', 'acceptance_rate', 'graduation_rate_150pct', 'bachelors_4yr_completion_rate', 'retention_rate', 'student_faculty_ratio', 'undergraduate_enrollment', 'average_net_price', 'DEBT_MDN', 'GRAD_DEBT_MDN', 'MD_EARN_WNE_P10', 'MD_EARN_WNE_P6']

🏫 INSTITUTION TYPES:
institution_type
Private nonprofit 4-year     1618
Public 4-year                 848
Private for-profit 4-year     330
Name: count, dtype: int64

🏛️ COLLEGE SECTORS:
sector
2    1618
1     848
3     330
Name: count, dtype: int64

📊 COMPARISON POOL:
Original colleges: 2796
Currently rankable: 1709
Excluded from pool: 1087


In [46]:

def filter_ranked_colleges(
    rankings,
    states=None,
    institution_types=None,
    min_enrollment=None,
    max_average_net_price=None
):
    """
    Filter already-scored colleges.

    Scores stay unchanged.
    Displayed ranks update within the filtered results.
    """

    df = rankings.copy()

    # Filter by state
    if states:
        df = df[df["state"].isin(states)]

    # Filter by institution type
    if institution_types:
        df = df[
            df["institution_type"].isin(institution_types)
        ]

    # Filter by undergraduate enrollment
    if min_enrollment is not None:
        df = df[
            df["undergraduate_enrollment"] >= min_enrollment
        ]

    # Optional average net price cutoff
    if max_average_net_price is not None:
        df = df[
            df["average_net_price"] <= max_average_net_price
        ]

    df = df.copy()

    # Preserve the overall ranking
    df = df.rename(
        columns={"rank": "overall_rank"}
    )

    # New rank among filtered colleges
    df = df.reset_index(drop=True)
    df.insert(
        0,
        "filtered_rank",
        range(1, len(df) + 1)
    )

    return df


print("✅ College filtering function created!")


✅ College filtering function created!


In [47]:

filtered_results = filter_ranked_colleges(
    backend_result["ai_rankings"],

    states=["MA", "NY"],

    institution_types=[
        "Public 4-year",
        "Private nonprofit 4-year"
    ],

    min_enrollment=1000
)

print("Colleges matching filters:", len(filtered_results))

display(
    filtered_results[
        [
            "filtered_rank",
            "overall_rank",
            "college_name",
            "state",
            "institution_type",
            "final_score"
        ]
    ].head(10)
)


Colleges matching filters: 153


,filtered_rank,overall_rank,college_name,state,institution_type,final_score
0,1,3,CUNY Bernard M Baruch College,NY,Public 4-year,91.299330
1,2,5,Williams College,MA,Private nonprofit 4-year,89.771471
2,3,13,CUNY Hunter College,NY,Public 4-year,83.755933
3,4,15,CUNY City College,NY,Public 4-year,83.531630
4,5,22,CUNY John Jay College of Criminal Justice,NY,Public 4-year,81.636435
5,6,23,CUNY Queens College,NY,Public 4-year,80.706716
6,7,24,Wellesley College,MA,Private nonprofit 4-year,80.612444
7,8,27,Amherst College,MA,Private nonprofit 4-year,79.733112
8,9,30,Harvard University,MA,Private nonprofit 4-year,79.388531
9,10,33,Farmingdale State College,NY,Public 4-year,79.021195


In [48]:

original_scores = (
    backend_result["ai_rankings"]
    .set_index("unitid")["final_score"]
)

filtered_scores = (
    filtered_results
    .set_index("unitid")["final_score"]
)

assert filtered_scores.equals(
    original_scores.loc[filtered_scores.index]
)

assert filtered_results["state"].isin(
    ["MA", "NY"]
).all()

assert (
    filtered_results["undergraduate_enrollment"] >= 1000
).all()

print("✅ Filters work correctly!")
print("✅ Ranking scores remain unchanged!")


✅ Filters work correctly!
✅ Ranking scores remain unchanged!


In [49]:

import pandas as pd
import numpy as np
import math

def prepare_reference_pool(colleges, catalog):
    df = colleges.copy()

    df["undergraduate_enrollment"] = pd.to_numeric(
        df["undergraduate_enrollment"],
        errors="coerce"
    )

    # Keep colleges with reported undergraduates
    df = df[
        df["undergraduate_enrollment"] > 0
    ].copy()

    # Convert ranking metrics into numbers
    for metric in catalog:
        df[metric] = pd.to_numeric(
            df[metric],
            errors="coerce"
        )

    # Remove invalid negative net prices
    if "average_net_price" in df.columns:
        df.loc[
            df["average_net_price"] < 0,
            "average_net_price"
        ] = np.nan

    return df.reset_index(drop=True)


reference_pool = prepare_reference_pool(
    colleges,
    rankable_metrics
)

print("Original colleges:", len(colleges))
print("Old comparison pool:", len(comparison_pool))
print("New reference pool:", len(reference_pool))

print("\nMissing values by metric:")
print(
    reference_pool[list(rankable_metrics)]
    .isna()
    .sum()
    .sort_values(ascending=False)
)


Original colleges: 2796
Old comparison pool: 1709
New reference pool: 2490

Missing values by metric:
bachelors_4yr_completion_rate    496
retention_rate                   417
GRAD_DEBT_MDN                    353
DEBT_MDN                         294
MD_EARN_WNE_P6                   230
MD_EARN_WNE_P10                  226
graduation_rate_150pct           207
average_net_price                205
student_faculty_ratio              1
dtype: int64


In [50]:

def rank_colleges_v2(pool, selected_weights, catalog):

    # 1. Validate selected metrics and weights
    active = {}

    for metric, raw_weight in selected_weights.items():
        if metric not in catalog:
            raise ValueError(f"Unknown metric: {metric}")

        if metric not in pool.columns:
            raise ValueError(f"Missing column: {metric}")

        direction = catalog[metric]["direction"]

        if direction not in ("higher", "lower"):
            raise ValueError(f"Unsupported direction: {metric}")

        if isinstance(raw_weight, bool):
            raise ValueError(f"Invalid weight: {metric}")

        try:
            weight = float(raw_weight)
        except (TypeError, ValueError):
            raise ValueError(f"Invalid weight: {metric}")

        if not math.isfinite(weight) or weight < 0:
            raise ValueError(f"Invalid weight: {metric}")

        if weight > 0:
            active[metric] = weight

    if not active:
        raise ValueError("Select at least one metric.")

    # 2. Normalize weights
    total = sum(active.values())

    normalized = {
        metric: weight / total
        for metric, weight in active.items()
    }

    df = pool.copy()

    # 3. Identify schools missing selected metrics
    missing = df[list(active)].isna()

    complete_mask = ~missing.any(axis=1)

    ranked = df.loc[complete_mask].copy()
    incomplete = df.loc[~complete_mask].copy()

    if ranked.empty:
        raise ValueError(
            "No colleges have all selected metrics."
        )

    # 4. Calculate scores using the reference pool
    ranked["final_score"] = 0.0

    for metric, weight in normalized.items():

        direction = catalog[metric]["direction"]

        # Reference percentiles include every school
        # with valid data for this particular metric.
        reference_percentiles = df[metric].rank(
            pct=True,
            ascending=(direction == "higher")
        ) * 100

        ranked[f"{metric}_percentile"] = (
            reference_percentiles.loc[ranked.index]
        )

        contribution = (
            ranked[f"{metric}_percentile"] * weight
        )

        ranked[f"{metric}_contribution"] = contribution
        ranked["final_score"] += contribution

    # 5. Sort ranked colleges
    ranked = ranked.sort_values(
        ["final_score", "college_name"],
        ascending=[False, True]
    ).reset_index(drop=True)

    ranked["rank"] = range(1, len(ranked) + 1)

    # 6. Track incomplete college information
    incomplete["missing_metrics"] = missing.loc[
        ~complete_mask
    ].apply(
        lambda row: [
            metric for metric in active
            if row[metric]
        ],
        axis=1
    )

    coverage = pd.Series(0.0, index=df.index)

    for metric, weight in normalized.items():
        coverage += (
            df[metric].notna().astype(float) * weight
        )

    incomplete["data_coverage_pct"] = (
        coverage.loc[~complete_mask] * 100
    )

    incomplete = incomplete.sort_values(
        ["data_coverage_pct", "college_name"],
        ascending=[False, True]
    ).reset_index(drop=True)

    return ranked, incomplete


print("✅ Ranking Engine V2 created!")


✅ Ranking Engine V2 created!


In [51]:

# Rankings using Agent 2's approved weights
ai_v2, ai_incomplete = rank_colleges_v2(
    reference_pool,
    weights,
    rankable_metrics
)

# Rankings with manually adjusted weights
custom_v2, custom_incomplete = rank_colleges_v2(
    reference_pool,
    custom_weights,
    rankable_metrics
)

print("📊 RANKING COVERAGE")
print("Reference colleges:", len(reference_pool))
print("Old complete pool:", len(comparison_pool))
print("New fully scored colleges:", len(ai_v2))
print("Colleges needing more data:", len(ai_incomplete))

print("\n🏆 TOP 10 — AI PREFERENCES")
display(
    ai_v2[
        ["rank", "college_name", "state", "final_score"]
    ].head(10)
)

print("\n🎯 TOP 10 — CUSTOM PREFERENCES")
display(
    custom_v2[
        ["rank", "college_name", "state", "final_score"]
    ].head(10)
)

print("\n🔎 EXAMPLES OF INCOMPLETE COLLEGES")
display(
    ai_incomplete[
        [
            "college_name",
            "state",
            "missing_metrics",
            "data_coverage_pct"
        ]
    ].head(10)
)

# Both scenarios use the same four metrics,
# so changing their positive weights alone
# should not change which schools are eligible.
assert set(ai_v2["unitid"]) == set(custom_v2["unitid"])

assert len(ai_v2) + len(ai_incomplete) == len(reference_pool)

assert ai_v2["final_score"].between(0, 100).all()
assert custom_v2["final_score"].between(0, 100).all()

print("\n✅ Missing-data handling checks passed!")


📊 RANKING COVERAGE
Reference colleges: 2490
Old complete pool: 1709
New fully scored colleges: 1988
Colleges needing more data: 502

🏆 TOP 10 — AI PREFERENCES


,rank,college_name,state,final_score
0,1,United States Merchant Marine Academy,NY,96.342547
1,2,CUNY Bernard M Baruch College,NY,89.841972
2,3,California Institute of Technology,CA,87.608750
3,4,University of North Carolina at Chapel Hill,NC,86.043003
4,5,Williams College,MA,85.402878
5,6,CUNY Hunter College,NY,84.256122
6,7,CUNY City College,NY,84.078426
7,8,Princeton University,NJ,83.159614
8,9,University of Washington-Seattle Campus,WA,83.010820
9,10,CUNY John Jay College of Criminal Justice,NY,82.379514



🎯 TOP 10 — CUSTOM PREFERENCES


,rank,college_name,state,final_score
0,1,California Institute of Technology,CA,96.327882
1,2,United States Merchant Marine Academy,NY,96.115090
2,3,Princeton University,NJ,95.066333
3,4,Williams College,MA,93.455757
4,5,Stanford University,CA,92.682464
5,6,Massachusetts Institute of Technology,MA,92.658194
6,7,University of California-Berkeley,CA,92.650311
7,8,Yale University,CT,92.631417
8,9,Harvard University,MA,92.499198
9,10,Dartmouth College,NH,92.078655



🔎 EXAMPLES OF INCOMPLETE COLLEGES


,college_name,state,missing_metrics,data_coverage_pct
0,Abilene Christian University-Undergraduate Online,TX,[graduation_rate_150pct],81.25
1,Arizona College of Nursing - Chesapeake,VA,[graduation_rate_150pct],81.25
2,Arizona College of Nursing - Hartford,CT,[graduation_rate_150pct],81.25
3,Arizona College of Nursing-Aurora,CO,[graduation_rate_150pct],81.25
4,Arizona College of Nursing-Cincinnati,OH,[graduation_rate_150pct],81.25
5,Arizona College of Nursing-Cleveland,OH,[graduation_rate_150pct],81.25
6,Arizona College of Nursing-Dallas,TX,[graduation_rate_150pct],81.25
7,Arizona College of Nursing-Falls Church,VA,[graduation_rate_150pct],81.25
8,Arizona College of Nursing-Ontario,CA,[graduation_rate_150pct],81.25
9,Arizona College of Nursing-Phoenix,AZ,[graduation_rate_150pct],81.25



✅ Missing-data handling checks passed!


In [52]:

simple_weights = {
    "average_net_price": 5,
    "graduation_rate_150pct": 5
}

simple_rankings, simple_incomplete = rank_colleges_v2(
    reference_pool,
    simple_weights,
    rankable_metrics
)

print("🏫 Two-metric college rankings")
print("Fully scored:", len(simple_rankings))
print("Needs more data:", len(simple_incomplete))

display(
    simple_rankings[
        ["rank", "college_name", "final_score"]
    ].head(10)
)


🏫 Two-metric college rankings
Fully scored: 2217
Needs more data: 273


,rank,college_name,final_score
0,1,Divine Word College,99.233810
1,2,United States Merchant Marine Academy,94.656844
2,3,Sattler College,91.527473
3,4,University of North Carolina at Chapel Hill,90.982938
4,5,CUNY Bernard M Baruch College,90.397194
5,6,Yeshivath Beth Moshe,90.284138
6,7,Rabbinical College Bobover Yeshiva Bnei Zion,90.171355
7,8,Dewey University-Juana Díaz,90.058534
8,9,Western Technical College,89.784503
9,10,Yeshiva Gedola Tiferes Yerachmiel,88.969809


In [53]:
print("🏫 RANKMYCOLLEGE V2 — FINAL VERIFICATION\n")

print("Original colleges:", len(colleges))
print("Reference pool:", len(reference_pool))
print("Old comparison pool:", len(comparison_pool))

print("\n📊 FOUR-METRIC RANKING")
print("Fully ranked:", len(ai_v2))
print("Missing selected data:", len(ai_incomplete))

print("\n📊 TWO-METRIC RANKING")
print("Fully ranked:", len(simple_rankings))
print("Missing selected data:", len(simple_incomplete))

assert len(simple_rankings) >= len(ai_v2), \
    "Two-metric ranking should include at least as many colleges."

assert len(ai_v2) + len(ai_incomplete) == len(reference_pool)

assert len(simple_rankings) + len(simple_incomplete) == len(reference_pool)

assert ai_v2["unitid"].is_unique

assert ai_v2["final_score"].between(0, 100).all()

print("\n✅ ALL V2 CHECKS PASSED!")

🏫 RANKMYCOLLEGE V2 — FINAL VERIFICATION

Original colleges: 2796
Reference pool: 2490
Old comparison pool: 1709

📊 FOUR-METRIC RANKING
Fully ranked: 1988
Missing selected data: 502

📊 TWO-METRIC RANKING
Fully ranked: 2217
Missing selected data: 273

✅ ALL V2 CHECKS PASSED!


In [54]:

def prepare_explainer_evidence_v2(
    request, original, custom,
    original_weights, custom_weights, catalog
):
    original_lookup = original.set_index("unitid")
    custom_lookup = custom.set_index("unitid")

    # Gather the top 3 from each scenario
    college_ids = list(dict.fromkeys(
        original["unitid"].head(3).tolist()
        + custom["unitid"].head(3).tolist()
    ))

    metrics = list(dict.fromkeys(
        list(original_weights) + list(custom_weights)
    ))

    def normalize(w):
        active = {k: float(v) for k, v in w.items() if v > 0}
        total = sum(active.values())
        return {k: v / total for k, v in active.items()}

    def safe_number(value):
        if value is None or pd.isna(value):
            return None
        return float(value)

    college_evidence = []

    for unitid in college_ids:
        a = (
            original_lookup.loc[unitid]
            if unitid in original_lookup.index
            else None
        )

        b = (
            custom_lookup.loc[unitid]
            if unitid in custom_lookup.index
            else None
        )

        source = a if a is not None else b

        metric_evidence = {}

        for metric in metrics:
            metric_evidence[metric] = {
                "actual_value": safe_number(source[metric]),
                "direction": catalog[metric]["direction"],
                "ai_contribution": (
                    safe_number(a.get(f"{metric}_contribution"))
                    if a is not None else None
                ),
                "custom_contribution": (
                    safe_number(b.get(f"{metric}_contribution"))
                    if b is not None else None
                )
            }

        college_evidence.append({
            "college_name": str(source["college_name"]),
            "ai_rank": int(a["rank"]) if a is not None else None,
            "custom_rank": int(b["rank"]) if b is not None else None,
            "ai_score": safe_number(a["final_score"]) if a is not None else None,
            "custom_score": safe_number(b["final_score"]) if b is not None else None,
            "metrics": metric_evidence
        })

    return {
        "student_request": request,
        "ai_weights": normalize(original_weights),
        "custom_weights": normalize(custom_weights),
        "original_ranked_count": len(original),
        "custom_ranked_count": len(custom),
        "colleges": college_evidence,
        "comparison_rule": (
            "A null rank means the college was not fully scored "
            "in that scenario. Do not invent missing ranks, "
            "scores, metric values, or comparisons."
        )
    }

print("✅ Agent 3 evidence helper upgraded!")


✅ Agent 3 evidence helper upgraded!


In [55]:

def run_rankmycollege_v2(student_request, edited_weights=None):

    print("🧠 Agent 1: Understanding preferences...")
    initial_plan = preference_agent(student_request)

    print("🛡️ Agent 2: Auditing metric choices...")
    approved_plan = auditor_agent(
        student_request,
        initial_plan
    )

    ai_weights = validate_weights(approved_plan)

    selected_weights = (
        edited_weights if edited_weights is not None
        else ai_weights
    )

    print("📊 Ranking colleges with V2...")

    ai_rankings, ai_incomplete = rank_colleges_v2(
        reference_pool,
        ai_weights,
        rankable_metrics
    )

    custom_rankings, custom_incomplete = rank_colleges_v2(
        reference_pool,
        selected_weights,
        rankable_metrics
    )

    print("💬 Agent 3: Explaining results...")

    evidence = prepare_explainer_evidence_v2(
        student_request,
        ai_rankings,
        custom_rankings,
        ai_weights,
        selected_weights,
        rankable_metrics
    )

    explanation = explainer_agent(evidence)

    print("✅ Three-agent V2 pipeline completed!")

    return {
        "initial_plan": initial_plan,
        "approved_plan": approved_plan,
        "ai_weights": ai_weights,
        "ai_rankings": ai_rankings,
        "ai_incomplete": ai_incomplete,
        "custom_rankings": custom_rankings,
        "custom_incomplete": custom_incomplete,
        "explanation": explanation
    }


In [56]:

test_request = """
I want an affordable university with low student
debt, strong career earnings, and high graduation
rates. Prestige isn't important to me.
"""

edited_weights = {
    "average_net_price": 1,
    "DEBT_MDN": 1,
    "MD_EARN_WNE_P10": 8,
    "graduation_rate_150pct": 3
}

v2_result = run_rankmycollege_v2(
    test_request,
    edited_weights
)

print("\n📊 V2 DATA COVERAGE")
print("Reference pool:", len(reference_pool))
print("AI ranking:", len(v2_result["ai_rankings"]))
print("Custom ranking:", len(v2_result["custom_rankings"]))

print("\n🏆 ORIGINAL TOP 5")
display(
    v2_result["ai_rankings"][
        ["rank", "college_name", "final_score"]
    ].head()
)

print("\n🎯 CUSTOM TOP 5")
display(
    v2_result["custom_rankings"][
        ["rank", "college_name", "final_score"]
    ].head()
)

report = v2_result["explanation"]

print("\n💬 YOUR TOP MATCH")
print(report.original_winner_explanation)

print("\n🎯 YOUR CUSTOM TOP MATCH")
print(report.custom_winner_explanation)

print("\n🔄 WHAT CHANGED?")
print(report.why_rankings_changed)

print("\n⚖️ THE TRADEOFFS")
for item in report.key_tradeoffs:
    print("-", item)

assert (
    len(v2_result["ai_rankings"])
    + len(v2_result["ai_incomplete"])
    == len(reference_pool)
)

assert (
    len(v2_result["custom_rankings"])
    + len(v2_result["custom_incomplete"])
    == len(reference_pool)
)

print("\n✅ V2 BACKEND VALIDATION PASSED!")


🧠 Agent 1: Understanding preferences...
🛡️ Agent 2: Auditing metric choices...
📊 Ranking colleges with V2...
💬 Agent 3: Explaining results...
✅ Three-agent V2 pipeline completed!

📊 V2 DATA COVERAGE
Reference pool: 2490
AI ranking: 1988
Custom ranking: 1988

🏆 ORIGINAL TOP 5


,rank,college_name,final_score
0,1,United States Merchant Marine Academy,96.347416
1,2,CUNY Bernard M Baruch College,89.445717
2,3,California Institute of Technology,87.799235
3,4,Williams College,85.890536
4,5,University of North Carolina at Chapel Hill,85.376003



🎯 CUSTOM TOP 5


,rank,college_name,final_score
0,1,California Institute of Technology,96.327882
1,2,United States Merchant Marine Academy,96.115090
2,3,Princeton University,95.066333
3,4,Williams College,93.455757
4,5,Stanford University,92.682464



💬 YOUR TOP MATCH
United States Merchant Marine Academy ranked first because of its low average net price, low student debt, and solid graduation rate.

🎯 YOUR CUSTOM TOP MATCH
California Institute of Technology took the top spot because you prioritized high reported earnings 10 years after entering college.

🔄 WHAT CHANGED?
Shifting your focus toward earnings pushed California Institute of Technology to first place, outweighing its higher average net price.

⚖️ THE TRADEOFFS
- United States Merchant Marine Academy has a lower average net price than California Institute of Technology.
- California Institute of Technology has higher reported earnings 10 years after entering college than United States Merchant Marine Academy.

✅ V2 BACKEND VALIDATION PASSED!


In [57]:

import numpy as np
import pandas as pd

print("🧪 RANKMYCOLLEGE BACKEND TESTS\n")

# Test 1: Every college is accounted for
ranked, incomplete = rank_colleges_v2(
    reference_pool, weights, rankable_metrics
)

assert len(ranked) + len(incomplete) == len(reference_pool)
print("✅ TEST 1: All colleges accounted for")


# Test 2: No duplicate universities
assert ranked["unitid"].is_unique
assert incomplete["unitid"].is_unique

assert set(ranked["unitid"]).isdisjoint(
    set(incomplete["unitid"])
)
print("✅ TEST 2: No duplicate or overlapping colleges")


# Test 3: Scores are valid
assert ranked["final_score"].between(0, 100).all()
assert ranked["final_score"].notna().all()
print("✅ TEST 3: All scores between 0 and 100")


# Test 4: Contributions add up correctly
active_metrics = [
    metric for metric, weight in weights.items()
    if weight > 0
]

contribution_columns = [
    f"{metric}_contribution"
    for metric in active_metrics
]

np.testing.assert_allclose(
    ranked[contribution_columns].sum(axis=1),
    ranked["final_score"],
    atol=1e-8
)
print("✅ TEST 4: Mathematical calculations correct")


# Test 5: Scaling weights doesn't change rankings
scaled_weights = {
    metric: weight * 10
    for metric, weight in weights.items()
}

scaled, _ = rank_colleges_v2(
    reference_pool, scaled_weights, rankable_metrics
)

assert ranked["unitid"].tolist() == scaled["unitid"].tolist()

np.testing.assert_allclose(
    ranked["final_score"],
    scaled["final_score"],
    atol=1e-8
)
print("✅ TEST 5: Weight normalization works")


# Test 6: Fewer selected metrics improve coverage
four_metrics = {
    "average_net_price": 5,
    "DEBT_MDN": 4,
    "MD_EARN_WNE_P10": 4,
    "graduation_rate_150pct": 3
}

two_metrics = {
    "average_net_price": 5,
    "graduation_rate_150pct": 5
}

four_ranked, _ = rank_colleges_v2(
    reference_pool, four_metrics, rankable_metrics
)

two_ranked, _ = rank_colleges_v2(
    reference_pool, two_metrics, rankable_metrics
)

assert set(four_ranked["unitid"]).issubset(
    set(two_ranked["unitid"])
)

print("✅ TEST 6: Missing-data handling works")


# Test 7: Invalid weights are rejected
def expect_invalid(bad_weights):
    try:
        rank_colleges_v2(
            reference_pool, bad_weights, rankable_metrics
        )
    except ValueError:
        return
    raise AssertionError(
        f"Invalid weights were accepted: {bad_weights}"
    )

expect_invalid({"average_net_price": -5})
expect_invalid({"average_net_price": float("nan")})
expect_invalid({"made_up_metric": 5})
expect_invalid({"average_net_price": 0})

print("✅ TEST 7: Invalid AI/user inputs rejected")


# Test 8: Rankings are reproducible
repeat, _ = rank_colleges_v2(
    reference_pool, weights, rankable_metrics
)

pd.testing.assert_frame_equal(
    ranked[["unitid", "rank", "final_score"]],
    repeat[["unitid", "rank", "final_score"]]
)

print("✅ TEST 8: Rankings reproducible")

print("\n🎉 ALL BACKEND TESTS PASSED!")


🧪 RANKMYCOLLEGE BACKEND TESTS

✅ TEST 1: All colleges accounted for
✅ TEST 2: No duplicate or overlapping colleges
✅ TEST 3: All scores between 0 and 100
✅ TEST 4: Mathematical calculations correct
✅ TEST 5: Weight normalization works
✅ TEST 6: Missing-data handling works
✅ TEST 7: Invalid AI/user inputs rejected
✅ TEST 8: Rankings reproducible

🎉 ALL BACKEND TESTS PASSED!


In [58]:
test_request = """
I want an affordable university with low student
debt, strong career earnings, and high graduation
rates. Prestige isn't important to me.
"""


v2_result = run_rankmycollege_v2(
    test_request,
    edited_weights
)

print("\n📊 V2 DATA COVERAGE")
print("Reference pool:", len(reference_pool))
print("AI ranking:", len(v2_result["ai_rankings"]))
print("Custom ranking:", len(v2_result["custom_rankings"]))

🧠 Agent 1: Understanding preferences...
🛡️ Agent 2: Auditing metric choices...
📊 Ranking colleges with V2...
💬 Agent 3: Explaining results...
✅ Three-agent V2 pipeline completed!

📊 V2 DATA COVERAGE
Reference pool: 2490
AI ranking: 1956
Custom ranking: 1988


In [59]:
test_request = """
I want to find a school that give excellent financial aid and amazing data science program. Also i want the school with very high prestigious reputation for employment opprtunity.
"""

v2_result = run_rankmycollege_v2(
    test_request,
    edited_weights
)

print("\n📊 V2 DATA COVERAGE")
print("Reference pool:", len(reference_pool))
print("AI ranking:", len(v2_result["ai_rankings"]))
print("Custom ranking:", len(v2_result["custom_rankings"]))

print("\n🏆 ORIGINAL TOP 5")
display(
    v2_result["ai_rankings"][
        ["rank", "college_name", "final_score"]
    ].head()
)

🧠 Agent 1: Understanding preferences...
🛡️ Agent 2: Auditing metric choices...
📊 Ranking colleges with V2...
💬 Agent 3: Explaining results...
✅ Three-agent V2 pipeline completed!

📊 V2 DATA COVERAGE
Reference pool: 2490
AI ranking: 2032
Custom ranking: 1988

🏆 ORIGINAL TOP 5


,rank,college_name,final_score
0,1,United States Merchant Marine Academy,97.686367
1,2,CUNY Bernard M Baruch College,91.982160
2,3,CUNY City College,87.643403
3,4,CUNY Hunter College,87.559366
4,5,California Institute of Technology,87.031870


In [60]:

print("🧠 AGENT 1 — INTERPRETATION")
print(
    v2_result["initial_plan"].model_dump_json(indent=2)
)

print("\n🛡️ AGENT 2 — FINAL INTERPRETATION")
print(
    v2_result["approved_plan"].model_dump_json(indent=2)
)

print("\n📊 FINAL AI WEIGHTS")
for metric, weight in v2_result["ai_weights"].items():
    print(f"{metric}: {weight:.1%}")

print("\n⚠️ UNSUPPORTED STUDENT PRIORITIES")
for item in v2_result["approved_plan"].unsupported:
    print("-", item)

print("\n🏆 AI TOP 10")
display(
    v2_result["ai_rankings"][
        ["rank", "college_name", "final_score"]
    ].head(10)
)


🧠 AGENT 1 — INTERPRETATION
{
  "choices": [
    {
      "metric": "average_net_price",
      "weight": 5.0,
      "explanation": "Measures the average net cost after grant aid to address your preference for excellent financial aid."
    },
    {
      "metric": "MD_EARN_WNE_P10",
      "weight": 4.0,
      "explanation": "Tracks median earnings 10 years after entry as a proxy for strong long-term employment opportunities and reputation."
    },
    {
      "metric": "DEBT_MDN",
      "weight": 3.0,
      "explanation": "Evaluates median federal student loan debt, helping identify schools with manageable debt burdens alongside financial aid."
    },
    {
      "metric": "MD_EARN_WNE_P6",
      "weight": 2.0,
      "explanation": "Assesses median earnings 6 years after entry to gauge early-career employment success."
    },
    {
      "metric": "retention_rate",
      "weight": 1.0,
      "explanation": "Reflects overall student satisfaction and institutional quality."
    }
  ],
  "un

,rank,college_name,final_score
0,1,United States Merchant Marine Academy,97.686367
1,2,CUNY Bernard M Baruch College,91.982160
2,3,CUNY City College,87.643403
3,4,CUNY Hunter College,87.559366
4,5,California Institute of Technology,87.031870
5,6,Western Governors University,86.490892
6,7,CUNY John Jay College of Criminal Justice,85.318272
7,8,CUNY Queens College,85.181669
8,9,University of North Carolina at Chapel Hill,84.801166
9,10,Mayo Clinic College of Medicine and Science,84.249479


In [61]:

def show_recommendation_coverage(result):
    plan = result["approved_plan"]
    unsupported = plan.unsupported

    if unsupported:
        print("⚠️ PARTIAL MATCH — NOT ALL PRIORITIES EVALUATED")
        print("\nRankings currently use these metrics:")

        for metric, weight in result["ai_weights"].items():
            print(f"  {metric}: {weight:.1%}")

        print("\nStudent priorities NOT evaluated:")
        for item in unsupported:
            print("  -", item)

        print(
            "\nThese are the highest-ranked colleges "
            "based only on the available metrics, "
            "not complete matches to every requirement."
        )
    else:
        print("✅ All interpreted priorities are covered.")

    print("\n🏆 TOP 5 BASED ON AVAILABLE DATA")
    display(
        result["ai_rankings"][
            ["rank", "college_name", "final_score"]
        ].head(5)
    )

show_recommendation_coverage(v2_result)


⚠️ PARTIAL MATCH — NOT ALL PRIORITIES EVALUATED

Rankings currently use these metrics:
  average_net_price: 35.7%
  MD_EARN_WNE_P10: 28.6%
  DEBT_MDN: 21.4%
  MD_EARN_WNE_P6: 14.3%

Student priorities NOT evaluated:
  - Amazing data science program
  - Very high prestigious reputation for employment

These are the highest-ranked colleges based only on the available metrics, not complete matches to every requirement.

🏆 TOP 5 BASED ON AVAILABLE DATA


,rank,college_name,final_score
0,1,United States Merchant Marine Academy,97.686367
1,2,CUNY Bernard M Baruch College,91.982160
2,3,CUNY City College,87.643403
3,4,CUNY Hunter College,87.559366
4,5,California Institute of Technology,87.031870


In [62]:

import pandas as pd
import zipfile
import urllib.request
from pathlib import Path

zip_path = Path("C2024_A.zip")

url = "https://nces.ed.gov/ipeds/datacenter/data/C2024_A.zip"

if not zip_path.exists():
    print("Downloading IPEDS program data...")
    urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path) as archive:
    print("Files inside ZIP:")
    print(archive.namelist())

print("✅ Program dataset downloaded!")


Files inside ZIP:
['C2024_a.csv']
✅ Program dataset downloaded!


In [63]:

# Academic programs we want to identify
program_codes = {
    "30.7001": "Data Science",
    "30.7099": "Data Science",
    "30.7101": "Data Analytics"
}

parts = []

with zipfile.ZipFile("C2024_A.zip") as archive:

    csv_name = next(
        name for name in archive.namelist()
        if Path(name).name.upper() == "C2024_A.CSV"
    )

    with archive.open(csv_name) as csv_file:

        chunks = pd.read_csv(
            csv_file,
            usecols=lambda c: c.upper() in {
                "UNITID", "CIPCODE", "AWLEVEL",
                "MAJORNUM", "CTOTALT"
            },
            dtype={"CIPCODE": "string"},
            chunksize=200000
        )

        for chunk in chunks:
            chunk.columns = chunk.columns.str.upper()

            chunk["CIPCODE"] = (
                chunk["CIPCODE"].str.strip()
            )

            selected = chunk[
                chunk["CIPCODE"].isin(program_codes)
                & (chunk["AWLEVEL"] == 5)
                & (chunk["MAJORNUM"] == 1)
            ].copy()

            selected["CTOTALT"] = pd.to_numeric(
                selected["CTOTALT"],
                errors="coerce"
            )

            parts.append(selected)

program_data = pd.concat(
    parts, ignore_index=True
)

program_data = program_data[
    program_data["CTOTALT"] >= 0
].copy()

program_data["program"] = (
    program_data["CIPCODE"].map(program_codes)
)

print("✅ Data Science/Analytics records extracted!")
print("Records:", len(program_data))

display(program_data.head(10))


✅ Data Science/Analytics records extracted!
Records: 247


,UNITID,CIPCODE,MAJORNUM,AWLEVEL,CTOTALT,program
0,100751,30.7001,1,5,0,Data Science
1,104151,30.7001,1,5,47,Data Science
2,104586,30.7001,1,5,1,Data Science
3,105154,30.7101,1,5,0,Data Analytics
4,105330,30.7001,1,5,6,Data Science
5,106342,30.7001,1,5,3,Data Science
6,106704,30.7001,1,5,0,Data Science
7,107044,30.7001,1,5,3,Data Science
8,109785,30.7001,1,5,0,Data Science
9,110644,30.7001,1,5,2,Data Science


In [64]:

import pandas as pd
import zipfile
from pathlib import Path

parts = []

with zipfile.ZipFile("C2024_A.zip") as archive:

    csv_name = next(
        name for name in archive.namelist()
        if Path(name).name.upper() == "C2024_A.CSV"
    )

    with archive.open(csv_name) as csv_file:

        chunks = pd.read_csv(
            csv_file,
            usecols=lambda c: c.upper() in {
                "UNITID", "CIPCODE", "AWLEVEL",
                "MAJORNUM", "CTOTALT"
            },
            dtype=str,
            chunksize=200000
        )

        for chunk in chunks:
            chunk.columns = chunk.columns.str.upper()

            chunk["CIPCODE"] = chunk["CIPCODE"].str.strip()

            # Keep six-digit program classifications
            valid_cip = chunk["CIPCODE"].str.fullmatch(
                r"\d{2}\.\d{4}", na=False
            )

            # Exclude total-category codes
            valid_cip &= ~chunk["CIPCODE"].str.startswith(
                "99.", na=False
            )

            # Bachelor's degrees and first majors
            selected = chunk[
                (chunk["AWLEVEL"].str.strip() == "5")
                & (chunk["MAJORNUM"].str.strip() == "1")
                & valid_cip
            ].copy()

            selected["CTOTALT"] = pd.to_numeric(
                selected["CTOTALT"], errors="coerce"
            )

            selected["UNITID"] = pd.to_numeric(
                selected["UNITID"], errors="coerce"
            )

            selected = selected[
                selected["CTOTALT"].ge(0)
                & selected["UNITID"].notna()
            ]

            parts.append(
                selected[["UNITID", "CIPCODE", "CTOTALT"]]
            )

# Combine all programs
all_programs = pd.concat(parts, ignore_index=True)

all_programs = (
    all_programs
    .groupby(["UNITID", "CIPCODE"], as_index=False)
    ["CTOTALT"]
    .sum()
)

all_programs = all_programs.rename(
    columns={
        "UNITID": "unitid",
        "CIPCODE": "cipcode",
        "CTOTALT": "bachelors_awarded_2024"
    }
)

all_programs["unitid"] = all_programs["unitid"].astype(int)

# Keep institutions present in our college database
all_programs = all_programs[
    all_programs["unitid"].isin(colleges["unitid"])
].copy()

# Save the new program dataset
all_programs.to_csv(
    "college_programs_2024.csv",
    index=False
)

print("✅ ALL MAJORS IMPORTED!")
print("Unique colleges:", all_programs["unitid"].nunique())
print("Unique CIP program codes:", all_programs["cipcode"].nunique())
print("Total college-program records:", len(all_programs))

display(all_programs.head(10))


✅ ALL MAJORS IMPORTED!
Unique colleges: 2437
Unique CIP program codes: 1251
Total college-program records: 81869


,unitid,cipcode,bachelors_awarded_2024
0,100654,01.0999,12
1,100654,01.1001,10
2,100654,01.9999,2
3,100654,03.0599,0
4,100654,04.0301,8
5,100654,10.0202,32
6,100654,11.0101,25
7,100654,13.1001,0
8,100654,13.1012,11
9,100654,13.1202,6


In [65]:

import pandas as pd
import urllib.request
from pathlib import Path

cip_url = (
    "https://nces.ed.gov/ipeds/cipcode/"
    "Files/CIPCode2020.csv"
)

cip_path = Path("CIPCode2020.csv")

if not cip_path.exists():
    urllib.request.urlretrieve(cip_url, cip_path)

try:
    cip_raw = pd.read_csv(
        cip_path,
        dtype="string",
        encoding="utf-8-sig"
    )
except UnicodeDecodeError:
    cip_raw = pd.read_csv(
        cip_path,
        dtype="string",
        encoding="cp1252"
    )

cip_raw.columns = cip_raw.columns.str.strip()

print("✅ Official CIP catalog downloaded!")
print("Records:", len(cip_raw))
print("Columns:", cip_raw.columns.tolist())


✅ Official CIP catalog downloaded!
Records: 2848
Columns: ['CIPFamily', 'CIPCode', 'Action', 'TextChange', 'CIPTitle', 'CIPDefinition', 'CrossReferences', 'Examples']


In [66]:

# Clean the CIP codes safely
def normalize_cip(series):
    s = series.astype("string").str.strip()

    # Remove Excel-style ="..." wrappers
    s = s.str.replace(
        r'^="([^"]+)"$',
        r'\1',
        regex=True
    )

    # Convert 110701 into 11.0701 if needed
    s = s.str.replace(
        r'^(\d{2})(\d{4})$',
        r'\1.\2',
        regex=True
    )

    return s


cip_raw["cipcode"] = normalize_cip(
    cip_raw["CIPCode"]
)

# Keep individual six-digit programs only
cip_names = cip_raw.loc[
    cip_raw["cipcode"].str.fullmatch(
        r"\d{2}\.\d{4}", na=False
    ),
    ["cipcode", "CIPTitle"]
].copy()

cip_names = cip_names.rename(
    columns={"CIPTitle": "major_name"}
)

cip_names["major_name"] = (
    cip_names["major_name"]
    .str.strip()
    .str.rstrip(".")
)

cip_names = cip_names.drop_duplicates(
    subset="cipcode"
)

# Load the program file created in the previous step
programs = pd.read_csv(
    "college_programs_2024.csv",
    dtype={"cipcode": "string"}
)

programs["cipcode"] = normalize_cip(
    programs["cipcode"]
)

# Attach major names
programs_named = programs.merge(
    cip_names,
    on="cipcode",
    how="left",
    validate="many_to_one"
)

# Check the result
print("✅ MAJOR NAMES CONNECTED!")
print("Total records:", len(programs_named))
print(
    "Records without a major name:",
    programs_named["major_name"].isna().sum()
)

display(programs_named.head(10))

programs_named.to_csv(
    "college_programs_named.csv",
    index=False
)


✅ MAJOR NAMES CONNECTED!
Total records: 81869
Records without a major name: 0


,unitid,cipcode,bachelors_awarded_2024,major_name
0,100654,01.0999,12,"Animal Sciences, Other"
1,100654,01.1001,10,Food Science
2,100654,01.9999,2,Agricultural/Animal/Plant/Veterinary Science a...
3,100654,03.0599,0,"Forestry, Other"
4,100654,04.0301,8,"City/Urban, Community, and Regional Planning"
5,100654,10.0202,32,Radio and Television Broadcasting Technology/T...
6,100654,11.0101,25,"Computer and Information Sciences, General"
7,100654,13.1001,0,"Special Education and Teaching, General"
8,100654,13.1012,11,Education/Teaching of Individuals with Speech ...
9,100654,13.1202,6,Elementary Education and Teaching


In [67]:

# Clean the CIP codes safely
def normalize_cip(series):
    s = series.astype("string").str.strip()

    # Remove Excel-style ="..." wrappers
    s = s.str.replace(
        r'^="([^"]+)"$',
        r'\1',
        regex=True
    )

    # Convert 110701 into 11.0701 if needed
    s = s.str.replace(
        r'^(\d{2})(\d{4})$',
        r'\1.\2',
        regex=True
    )

    return s


cip_raw["cipcode"] = normalize_cip(
    cip_raw["CIPCode"]
)

# Keep individual six-digit programs only
cip_names = cip_raw.loc[
    cip_raw["cipcode"].str.fullmatch(
        r"\d{2}\.\d{4}", na=False
    ),
    ["cipcode", "CIPTitle"]
].copy()

cip_names = cip_names.rename(
    columns={"CIPTitle": "major_name"}
)

cip_names["major_name"] = (
    cip_names["major_name"]
    .str.strip()
    .str.rstrip(".")
)

cip_names = cip_names.drop_duplicates(
    subset="cipcode"
)

# Load the program file created in the previous step
programs = pd.read_csv(
    "college_programs_2024.csv",
    dtype={"cipcode": "string"}
)

programs["cipcode"] = normalize_cip(
    programs["cipcode"]
)

# Attach major names
programs_named = programs.merge(
    cip_names,
    on="cipcode",
    how="left",
    validate="many_to_one"
)

# Check the result
print("✅ MAJOR NAMES CONNECTED!")
print("Total records:", len(programs_named))
print(
    "Records without a major name:",
    programs_named["major_name"].isna().sum()
)

display(programs_named.head(10))

programs_named.to_csv(
    "college_programs_named.csv",
    index=False
)


✅ MAJOR NAMES CONNECTED!
Total records: 81869
Records without a major name: 0


,unitid,cipcode,bachelors_awarded_2024,major_name
0,100654,01.0999,12,"Animal Sciences, Other"
1,100654,01.1001,10,Food Science
2,100654,01.9999,2,Agricultural/Animal/Plant/Veterinary Science a...
3,100654,03.0599,0,"Forestry, Other"
4,100654,04.0301,8,"City/Urban, Community, and Regional Planning"
5,100654,10.0202,32,Radio and Television Broadcasting Technology/T...
6,100654,11.0101,25,"Computer and Information Sciences, General"
7,100654,13.1001,0,"Special Education and Teaching, General"
8,100654,13.1012,11,Education/Teaching of Individuals with Speech ...
9,100654,13.1202,6,Elementary Education and Teaching


In [68]:

print("🔎 COMPUTER SCIENCE")
display(search_majors("computer science"))

print("\n🔎 NURSING")
display(search_majors("nursing"))

print("\n🔎 PSYCHOLOGY")
display(search_majors("psychology"))

print("\n🔎 DATA SCIENCE")
display(search_majors("data science"))


🔎 COMPUTER SCIENCE


NameError: name 'search_majors' is not defined

In [69]:
import pandas as pd

# Load our major database
programs_named = pd.read_csv(
    "college_programs_named.csv",
    dtype={"cipcode": "string"}
)

programs_named["bachelors_awarded_2024"] = pd.to_numeric(
    programs_named["bachelors_awarded_2024"],
    errors="coerce"
)

# Keep programs with reported bachelor's graduates
reported_programs = programs_named[
    programs_named["bachelors_awarded_2024"] > 0
].copy()

# Create searchable major catalog
major_catalog = (
    reported_programs
    .dropna(subset=["major_name"])
    .groupby(["cipcode", "major_name"], as_index=False)
    .agg(
        colleges_reporting=("unitid", "nunique"),
        bachelors_awarded=("bachelors_awarded_2024", "sum")
    )
)

# Define our search function
def search_majors(keyword, limit=20):
    matches = major_catalog[
        major_catalog["major_name"].str.contains(
            keyword,
            case=False,
            regex=False,
            na=False
        )
    ]

    return matches.sort_values(
        "colleges_reporting",
        ascending=False
    ).head(limit).reset_index(drop=True)


print("✅ Major search engine created!")
print("Available majors:", len(major_catalog))

✅ Major search engine created!
Available majors: 1155


In [70]:
print("🔎 COMPUTER SCIENCE")
display(search_majors("computer science"))

print("\n🔎 NURSING")
display(search_majors("nursing"))

print("\n🔎 PSYCHOLOGY")
display(search_majors("psychology"))

print("\n🔎 DATA SCIENCE")
display(search_majors("data science"))

🔎 COMPUTER SCIENCE


,cipcode,major_name,colleges_reporting,bachelors_awarded
0,11.0701,Computer Science,718,44627
1,30.0801,Mathematics and Computer Science,55,1100
2,30.3901,Economics and Computer Science,6,71



🔎 NURSING


,cipcode,major_name,colleges_reporting,bachelors_awarded
0,51.3801,Registered Nursing/Registered Nurse,1112,147345
1,51.3899,"Registered Nursing, Nursing Administration, Nu...",43,2011
2,51.3808,Nursing Science,17,1275
3,51.3818,Nursing Practice,13,423
4,51.3802,Nursing Administration,9,715
5,51.1105,Pre-Nursing Studies,7,65
6,51.3203,Nursing Education,5,19
7,51.3999,"Practical Nursing, Vocational Nursing and Nurs...",4,13
8,51.3803,Adult Health Nurse/Nursing,4,159
9,51.3811,Public Health/Community Nurse/Nursing,3,1199



🔎 PSYCHOLOGY


,cipcode,major_name,colleges_reporting,bachelors_awarded
0,42.0101,"Psychology, General",1269,114398
1,42.2799,"Research and Experimental Psychology, Other",106,11877
2,42.2813,Applied Psychology,48,1531
3,42.2704,Experimental Psychology,34,3493
4,42.2803,Counseling Psychology,26,295
5,42.2703,Developmental and Child Psychology,25,489
6,42.2812,Forensic Psychology,24,877
7,42.9999,"Psychology, Other",23,346
8,30.1001,Biopsychology,21,189
9,42.2804,Industrial and Organizational Psychology,20,520



🔎 DATA SCIENCE


,cipcode,major_name,colleges_reporting,bachelors_awarded
0,30.7001,"Data Science, General",130,2016
1,30.7099,"Data Science, Other",2,7


In [71]:

def rank_colleges_by_major(cipcode, selected_weights):

    # Find colleges with reported graduates in this major
    matching_programs = reported_programs[
        reported_programs["cipcode"] == str(cipcode)
    ].copy()

    matching_ids = set(matching_programs["unitid"])

    if not matching_ids:
        raise ValueError(
            f"No reported bachelor's completions for CIP {cipcode}"
        )

    # Rank against the full reference population FIRST
    ranked, incomplete = rank_colleges_v2(
        reference_pool,
        selected_weights,
        rankable_metrics
    )

    # Filter to schools with evidence for the major
    major_ranked = ranked[
        ranked["unitid"].isin(matching_ids)
    ].copy()

    major_incomplete = incomplete[
        incomplete["unitid"].isin(matching_ids)
    ].copy()

    # Preserve overall ranking
    major_ranked = major_ranked.rename(
        columns={"rank": "overall_rank"}
    )

    major_ranked = major_ranked.reset_index(drop=True)

    # Assign rank within the selected major's matches
    major_ranked.insert(
        0,
        "major_rank",
        range(1, len(major_ranked) + 1)
    )

    # Attach reported bachelor's completion counts
    major_ranked = major_ranked.merge(
        matching_programs[
            ["unitid", "major_name", "bachelors_awarded_2024"]
        ],
        on="unitid",
        how="left",
        validate="one_to_one"
    )

    return major_ranked, major_incomplete


print("✅ Major-aware ranking function created!")


✅ Major-aware ranking function created!


In [72]:

# Computer Science, General
computer_science_cip = "11.0701"

cs_rankings, cs_incomplete = rank_colleges_by_major(
    computer_science_cip,
    weights
)

print("🎓 COMPUTER SCIENCE COLLEGE RANKINGS")

print("Fully ranked:", len(cs_rankings))
print("Missing selected metric data:", len(cs_incomplete))

display(
    cs_rankings[
        [
            "major_rank",
            "college_name",
            "state",
            "final_score",
            "bachelors_awarded_2024"
        ]
    ].head(15)
)


🎓 COMPUTER SCIENCE COLLEGE RANKINGS
Fully ranked: 702
Missing selected metric data: 16


,major_rank,college_name,state,final_score,bachelors_awarded_2024
0,1,California Institute of Technology,CA,87.608750,94
1,2,University of North Carolina at Chapel Hill,NC,86.043003,334
2,3,Williams College,MA,85.402878,56
3,4,CUNY Hunter College,NY,84.256122,287
4,5,CUNY City College,NY,84.078426,84
5,6,Princeton University,NJ,83.159614,213
6,7,University of Washington-Seattle Campus,WA,83.010820,515
7,8,California State University-Fullerton,CA,81.797470,449
8,9,CUNY Queens College,NY,81.536789,256
9,10,University of California-San Diego,CA,81.522716,415


In [73]:

# Verify all returned colleges match the selected major
cs_ids = set(
    reported_programs.loc[
        reported_programs["cipcode"] == "11.0701",
        "unitid"
    ]
)

assert set(cs_rankings["unitid"]).issubset(cs_ids)

# Verify rankings are still in descending score order
assert cs_rankings["final_score"].is_monotonic_decreasing

# Verify scores didn't change after filtering
original_scores = ai_v2.set_index("unitid")["final_score"]

# This comparison uses the same 'weights' as ai_v2
for _, college in cs_rankings.iterrows():
    assert abs(
        college["final_score"]
        - original_scores.loc[college["unitid"]]
    ) < 1e-8

print("✅ Major filtering works!")
print("✅ Ranking scores are unchanged!")
print("✅ No additional AI calls required!")


✅ Major filtering works!
✅ Ranking scores are unchanged!
✅ No additional AI calls required!


In [74]:

from pydantic import BaseModel
from google.genai import types
import json

class MajorAwarePlan(BaseModel):
    choices: list[MetricChoice]
    unsupported: list[str]
    cautions: list[str]
    requested_major: str | None = None

print("✅ Major-aware output structure created!")



✅ Major-aware output structure created!


In [75]:

def preference_agent_v3(student_request):
    instructions = """
    You are RankMyCollege's Preference Interpreter.

    Your audience is high school seniors.

    Your responsibilities:
    1. Understand the student's college preferences.
    2. Identify their desired college major, if mentioned.
    3. Select relevant available ranking metrics.
    4. Assign positive importance weights.
    5. Identify requirements that cannot be measured.

    MAJOR IDENTIFICATION:
    - If the student names a major, extract its name.
    - Examples: Computer Science, Nursing,
      Psychology, Data Science, Biology.
    - Return a short, standard major name.
    - Do not invent a major if none was mentioned.
    - Do not invent a university's program availability.
    - The major is a program filter, NOT a numerical
      ranking metric.

    METRIC RULES:
    - Only choose metrics from available_metrics.
    - Choose at most 6 relevant metrics.
    - Use positive importance weights.
    - Explain each selected metric.
    - Never invent statistics.
    - Do not equate acceptance rate with prestige.
    - University-wide earnings do not measure the
      quality of a specific academic department.
    - Retention rates are not direct evidence
      of student satisfaction.
    - Mark unmeasurable program quality,
      prestige, or employment reputation as
      unsupported where appropriate.
    - Average net price is not personalized aid.

    Return the MajorAwarePlan structure.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=json.dumps({
            "student_request": student_request,
            "available_metrics": rankable_metrics
        }),
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            response_mime_type="application/json",
            response_schema=MajorAwarePlan,
            temperature=0.1
        )
    )

    return MajorAwarePlan.model_validate_json(
        response.text
    )

print("✅ Agent 1 V3 created!")


✅ Agent 1 V3 created!


In [76]:

student_request_v3 = """
I want to find a school that gives excellent
financial aid and has an amazing Data Science
program. I also want a prestigious reputation
for employment opportunities.
"""

plan_v3 = preference_agent_v3(
    student_request_v3
)

print("🎓 DETECTED MAJOR:")
print(plan_v3.requested_major)

print("\n📊 SELECTED METRICS:")
for choice in plan_v3.choices:
    print(f"- {choice.metric}: {choice.weight}")

print("\n⚠️ UNSUPPORTED REQUIREMENTS:")
for item in plan_v3.unsupported:
    print("-", item)

print("\n🔎 MATCHING MAJORS IN DATABASE:")
if plan_v3.requested_major:
    display(
        search_majors(plan_v3.requested_major)
    )
else:
    print("No specific major requested.")


🎓 DETECTED MAJOR:
Data Science

📊 SELECTED METRICS:
- average_net_price: 5.0
- MD_EARN_WNE_P10: 4.0
- DEBT_MDN: 3.0

⚠️ UNSUPPORTED REQUIREMENTS:
- amazing Data Science program quality
- prestigious reputation for employment opportunities

🔎 MATCHING MAJORS IN DATABASE:


,cipcode,major_name,colleges_reporting,bachelors_awarded
0,30.7001,"Data Science, General",130,2016
1,30.7099,"Data Science, Other",2,7


In [77]:

def auditor_agent_v3(student_request, first_plan):
    instructions = """
    You are RankMyCollege's Preference Auditor.

    Review Agent 1's interpretation of a high
    school student's college preferences.

    RESPONSIBILITIES:
    1. Verify the requested academic major.
    2. Correct a misidentified major when necessary.
    3. Review selected metrics and importance weights.
    4. Remove irrelevant or unsupported metrics.
    5. Identify criteria the dataset cannot evaluate.
    6. Preserve the student's actual priorities.

    MAJOR RULES:
    - Only identify majors explicitly requested.
    - Keep the major separate from numerical metrics.
    - Do not invent college program availability.
    - Do not claim degree counts measure program quality.
    - Do not substitute a loosely related major.
    - Do not invent CIP codes.

    RANKING RULES:
    - Only select metrics from available_metrics.
    - All selected weights must be positive.
    - Select at most 6 relevant metrics.
    - Institution-wide earnings do not measure
      major-specific employment outcomes.
    - Acceptance rate is not a prestige score.
    - Net price is not personalized financial aid.
    - Remove metrics that lack a clear connection
      to the student's request.

    Return a corrected MajorAwarePlan.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=json.dumps({
            "student_request": student_request,
            "initial_plan": first_plan.model_dump(),
            "available_metrics": rankable_metrics
        }),
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            response_mime_type="application/json",
            response_schema=MajorAwarePlan,
            temperature=0.1
        )
    )

    return MajorAwarePlan.model_validate_json(
        response.text
    )

print("✅ Agent 2 V3 created!")


✅ Agent 2 V3 created!


In [78]:

def resolve_major(major_name):
    if not major_name:
        return pd.DataFrame(
            columns=[
                "cipcode",
                "major_name",
                "colleges_reporting"
            ]
        )

    name = major_name.strip().casefold()

    # Exact name and its General/Other classifications
    accepted_names = {
        name,
        f"{name}, general",
        f"{name}, other"
    }

    matches = major_catalog[
        major_catalog["major_name"]
        .astype("string")
        .str.strip()
        .str.casefold()
        .isin(accepted_names)
    ].copy()

    if matches.empty:
        suggestions = search_majors(major_name)

        raise ValueError(
            f"No exact match for '{major_name}'. "
            f"Possible matches: "
            f"{suggestions['major_name'].tolist()[:5]}"
        )

    return matches.reset_index(drop=True)


print("✅ Major resolver created!")


✅ Major resolver created!


In [79]:

# Audit the Data Science request from our last test
audited_v3 = auditor_agent_v3(
    student_request_v3,
    plan_v3
)

print("🛡️ AGENT 2 — VERIFIED MAJOR")
print(audited_v3.requested_major)

print("\n📊 APPROVED METRICS")
for choice in audited_v3.choices:
    print(f"{choice.metric}: {choice.weight}")

print("\n⚠️ UNSUPPORTED REQUIREMENTS")
for item in audited_v3.unsupported:
    print("-", item)

# Resolve official program classifications
matching_majors = resolve_major(
    audited_v3.requested_major
)

print("\n🎓 MATCHING ACADEMIC PROGRAMS")
display(matching_majors)

# Identify colleges with reported bachelor's completions
matched_codes = set(matching_majors["cipcode"])

matching_college_ids = set(
    reported_programs.loc[
        reported_programs["cipcode"].isin(matched_codes),
        "unitid"
    ]
)

print(
    "\n🏫 Unique colleges with program evidence:",
    len(matching_college_ids)
)

print("✅ Major verified and matched to database!")


🛡️ AGENT 2 — VERIFIED MAJOR
Data Science

📊 APPROVED METRICS
average_net_price: 5.0
DEBT_MDN: 3.0

⚠️ UNSUPPORTED REQUIREMENTS
- amazing Data Science program quality
- prestigious reputation for employment opportunities

🎓 MATCHING ACADEMIC PROGRAMS


,cipcode,major_name,colleges_reporting,bachelors_awarded
0,30.7001,"Data Science, General",130,2016
1,30.7099,"Data Science, Other",2,7



🏫 Unique colleges with program evidence: 132
✅ Major verified and matched to database!


In [80]:

import pandas as pd

def filter_rankings_by_major(ranked, incomplete, cipcodes):

    # Find programs with reported bachelor's awards
    matches = reported_programs[
        reported_programs["cipcode"].isin(cipcodes)
        & (reported_programs["bachelors_awarded_2024"] > 0)
    ].copy()

    # Aggregate if one university matches multiple CIP codes
    major_counts = (
        matches.groupby("unitid", as_index=False)
        ["bachelors_awarded_2024"].sum()
        .rename(columns={
            "bachelors_awarded_2024":
            "matching_major_awards_2024"
        })
    )

    matching_ids = set(major_counts["unitid"])

    # Preserve scores calculated against all colleges
    major_ranked = ranked[
        ranked["unitid"].isin(matching_ids)
    ].copy()

    major_incomplete = incomplete[
        incomplete["unitid"].isin(matching_ids)
    ].copy()

    # Preserve the original overall rank
    major_ranked = major_ranked.rename(
        columns={"rank": "overall_rank"}
    ).reset_index(drop=True)

    # Add rank within the selected major
    major_ranked.insert(
        0, "rank", range(1, len(major_ranked) + 1)
    )

    # Attach program evidence without changing scores
    major_ranked = major_ranked.merge(
        major_counts,
        on="unitid",
        how="left",
        validate="one_to_one",
        sort=False
    )

    return major_ranked, major_incomplete


print("✅ Major filtering connected to V2!")


✅ Major filtering connected to V2!


In [81]:

import json
from pydantic import BaseModel
from google.genai import types


# ==========================================
# AGENT 3 — RECOMMENDATION EXPLAINER V3
# ==========================================

class ExplanationReport(BaseModel):
    original_winner_explanation: str
    custom_winner_explanation: str
    why_rankings_changed: str
    key_tradeoffs: list[str]


def explainer_agent(evidence):

    instructions = """
    You are Agent 3, RankMyCollege's
    Recommendation Explainer.

    Your audience is high school seniors,
    typically ages 17–18.

    Your job is to explain personalized
    college rankings in a short, clear,
    accurate, and engaging way.

    =====================================
    YOUR RESPONSIBILITIES
    =====================================

    1. Explain why the original #1 college
       ranked first.

    2. Explain why the custom #1 college
       ranked first after changing weights.

    3. Explain how changing priorities
       affected the rankings.

    4. Identify meaningful tradeoffs
       between the ranked universities.

    5. Use the student's requested major
       as context when one is provided.

    =====================================
    WRITING STYLE
    =====================================

    - Write for high school seniors.
    - Sound like a helpful older student.
    - Be friendly, natural, and direct.
    - Avoid academic jargon.
    - Avoid corporate or marketing language.
    - Do not exaggerate.
    - Do not use unnecessary filler.
    - Use "you" and "your" naturally.
    - Always name the actual universities.
    - Never write generic phrases like
      "This school is a good choice."

    LENGTH REQUIREMENTS:

    - Original explanation: maximum 35 words.
    - Custom explanation: maximum 35 words.
    - Ranking changes: maximum 30 words.
    - Give up to 2 meaningful tradeoffs.
    - Each tradeoff: maximum 25 words.
    - Do not repeat the same point unnecessarily.

    AVOID PHRASES LIKE:

    - "Incredible value"
    - "Fantastic balance"
    - "Exceptional institution"
    - "Prestigious establishment"
    - "Soared past competitors"
    - "Post-entry earnings"

    =====================================
    MAJOR-AWARE EXPLANATIONS
    =====================================

    The student may have requested a
    specific academic major.

    If requested_major is provided:

    - Understand that the candidate colleges
      were filtered using reported bachelor's
      completions in matching CIP programs.

    - Reference the selected major naturally
      when useful.

    - Never claim a university has the
      best program in that major.

    - Never claim the program is excellent
      without direct supporting evidence.

    - Never treat bachelor's completion
      counts as a measure of program quality.

    - Never assume that universities with
      no reported completions lack the major.

    - Do not claim university-wide earnings
      represent earnings for that specific major.

    - Do not claim employer reputation,
      teaching quality, curriculum strength,
      or program rankings were measured
      unless explicitly supported.

    - If the student's main priority is
      unsupported, do not imply that the
      recommendation satisfies that priority.

    =====================================
    FINANCIAL AND CAREER ACCURACY
    =====================================

    - Average net price is an institutional
      average, not the student's personal cost.

    - Median student debt does not predict
      an individual student's future debt.

    - MD_EARN_WNE_P10 means reported earnings
      10 years after entering college.

    - MD_EARN_WNE_P6 means reported earnings
      6 years after entering college.

    - Never describe either earnings measure
      as graduate earnings.

    - Never promise employment, earnings,
      financial aid, or admission.

    - Do not claim that higher reported
      earnings guarantee a better career.

    =====================================
    EVIDENCE AND RANKING RULES
    =====================================

    - Use ONLY the evidence provided.

    - The original winner must have
      ai_rank equal to 1.

    - The custom winner must have
      custom_rank equal to 1.

    - Never invent colleges, scores,
      rankings, or statistics.

    - Never change ranking results.

    - Never calculate new ranking scores.

    - Only compare actual metric values
      when both values are supplied.

    - A null value means unavailable.
      Never guess the missing value.

    - A null rank means that university
      was not fully scored in that scenario.

    - Ranking scores indicate performance
      under the selected weights, not
      overall college quality.

    - If original and custom winners are
      the same, explain that they stayed
      first rather than inventing a change.

    - When explaining a ranking change,
      focus on the actual weight changes.

    - Mention unsupported requirements
      only when necessary to prevent a
      misleading recommendation.

    - Do not create a separate
      data limitations section.

    =====================================
    OUTPUT REQUIREMENTS
    =====================================

    Return exactly these four fields:

    original_winner_explanation:
    Explain the original top college.

    custom_winner_explanation:
    Explain the customized top college.

    why_rankings_changed:
    Explain the effect of adjusting weights.

    key_tradeoffs:
    List up to 2 concise, evidence-supported
    comparisons involving named colleges.

    Keep everything short, personalized,
    and easy to understand.
    """

    response = client.models.generate_content(
        model=MODEL,
        contents=json.dumps(
            evidence,
            allow_nan=False
        ),
        config=types.GenerateContentConfig(
            system_instruction=instructions,
            response_mime_type="application/json",
            response_schema=ExplanationReport,
            temperature=0.1
        )
    )

    if response.parsed is not None:
        return ExplanationReport.model_validate(
            response.parsed
        )

    return ExplanationReport.model_validate_json(
        response.text
    )


print("✅ Agent 3 V3 — Recommendation Explainer ready!")


✅ Agent 3 V3 — Recommendation Explainer ready!


In [82]:

def run_rankmycollege_v3(student_request, edited_weights=None):

    print("🧠 Agent 1: Understanding preferences...")
    initial = preference_agent_v3(student_request)

    print("🛡️ Agent 2: Auditing preferences...")
    approved = auditor_agent_v3(
        student_request, initial
    )

    ai_weights = validate_weights(approved)

    selected_weights = (
        edited_weights if edited_weights is not None
        else ai_weights
    )

    # Resolve the requested major using our actual database
    major_name = approved.requested_major
    matched_programs = None

    if major_name:
        print(f"🎓 Looking for major: {major_name}")

        matched_programs = resolve_major(major_name)

        cipcodes = set(
            matched_programs["cipcode"].astype(str)
        )

    # Calculate scores against the full reference pool
    print("📊 Calculating personalized rankings...")

    ai_ranked, ai_incomplete = rank_colleges_v2(
        reference_pool, ai_weights, rankable_metrics
    )

    custom_ranked, custom_incomplete = rank_colleges_v2(
        reference_pool, selected_weights, rankable_metrics
    )

    # Apply major eligibility to both scenarios
    if major_name:

        ai_ranked, ai_incomplete = filter_rankings_by_major(
            ai_ranked, ai_incomplete, cipcodes
        )

        custom_ranked, custom_incomplete = filter_rankings_by_major(
            custom_ranked, custom_incomplete, cipcodes
        )

        if ai_ranked.empty or custom_ranked.empty:
            raise ValueError(
                "No fully scorable colleges found for "
                f"the selected major: {major_name}"
            )

    print("💬 Agent 3: Explaining recommendations...")

    evidence = prepare_explainer_evidence_v2(
        student_request,
        ai_ranked,
        custom_ranked,
        ai_weights,
        selected_weights,
        rankable_metrics
    )

    evidence["requested_major"] = major_name
    evidence["unsupported_priorities"] = approved.unsupported
    evidence["program_evidence_type"] = (
        "Reported 2024 bachelor's completions by CIP code; "
        "not a measure of program quality or proof that "
        "other colleges do not offer the major."
    )

    report = explainer_agent(evidence)

    print("✅ RankMyCollege V3 completed!")

    return {
        "major": major_name,
        "matched_programs": matched_programs,
        "approved_plan": approved,
        "ai_weights": ai_weights,
        "ai_rankings": ai_ranked,
        "ai_incomplete": ai_incomplete,
        "custom_rankings": custom_ranked,
        "custom_incomplete": custom_incomplete,
        "explanation": report
    }


In [83]:

test_request = """
I want a university with excellent financial aid,
an amazing Data Science program, and strong
employment opportunities.
"""

# For the comparison, simulate a student
# increasing the importance of reported earnings.
test_edited_weights = {
    "average_net_price": 2,
    "MD_EARN_WNE_P10": 7,
    "DEBT_MDN": 1
}

v3_result = run_rankmycollege_v3(
    test_request,
    test_edited_weights
)

print("\n🎓 DETECTED MAJOR")
print(v3_result["major"])

print("\n🏆 TOP 10 MAJOR-AWARE MATCHES")

display(
    v3_result["ai_rankings"][
        [
            "rank",
            "college_name",
            "state",
            "final_score",
            "matching_major_awards_2024"
        ]
    ].head(10)
)

print("\n🎯 TOP 5 AFTER ADJUSTING WEIGHTS")

display(
    v3_result["custom_rankings"][
        ["rank", "college_name", "final_score"]
    ].head(5)
)

print("\n💬 WHY YOUR TOP MATCH RANKED FIRST")
print(
    v3_result["explanation"].original_winner_explanation
)

print("\n⚠️ PRIORITIES NOT DIRECTLY MEASURED")
for item in v3_result["approved_plan"].unsupported:
    print("-", item)

print("\n✅ V3 TEST FINISHED!")


🧠 Agent 1: Understanding preferences...
🛡️ Agent 2: Auditing preferences...
🎓 Looking for major: Data Science
📊 Calculating personalized rankings...
💬 Agent 3: Explaining recommendations...
✅ RankMyCollege V3 completed!

🎓 DETECTED MAJOR
Data Science

🏆 TOP 10 MAJOR-AWARE MATCHES


,rank,college_name,state,final_score,matching_major_awards_2024
0,1,Utah Valley University,UT,79.819579,4
1,2,The University of Texas at Arlington,TX,77.147208,8
2,3,University of California-Davis,CA,74.410733,2
3,4,Florida Atlantic University,FL,72.835858,12
4,5,Rhode Island College,RI,70.528090,4
5,6,University of Illinois Urbana-Champaign,IL,69.626300,20
6,7,The University of Texas at Dallas,TX,69.322669,35
7,8,Portland State University,OR,69.242434,17
8,9,Dartmouth College,NH,69.062696,17
9,10,University of Wisconsin-Madison,WI,68.890419,246



🎯 TOP 5 AFTER ADJUSTING WEIGHTS


,rank,college_name,final_score
0,1,Dartmouth College,85.065066
1,2,University of California-Davis,84.527786
2,3,Stanford University,84.442624
3,4,University of Illinois Urbana-Champaign,83.121402
4,5,New Jersey Institute of Technology,81.665606



💬 WHY YOUR TOP MATCH RANKED FIRST
Utah Valley University ranked first initially because it offered the lowest average net price and manageable median debt among your options.

⚠️ PRIORITIES NOT DIRECTLY MEASURED
- Amazing Data Science program quality
- Employment outcomes specifically for Data Science

✅ V3 TEST FINISHED!


In [84]:

rankings = v3_result["ai_rankings"]
ai_weights = v3_result["ai_weights"]

print("🧠 WHAT THE AI ACTUALLY PRIORITIZED\n")

for metric, weight in ai_weights.items():
    print(f"{metric}: {weight:.1%}")

print("\n🏆 TOP 10 — ACTUAL COLLEGE DATA")

columns = [
    "rank",
    "college_name",
    "matching_major_awards_2024",
    "final_score"
] + list(ai_weights)

display(rankings[columns].head(10).round(2))

print("\n📊 HOW THE TOP 5 EARNED THEIR SCORES")

contributions = [
    f"{metric}_contribution"
    for metric in ai_weights
]

display(
    rankings[
        ["college_name"] + contributions
    ].head(5).round(2)
)

print("\n⚠️ REQUIREMENTS NOT MEASURED")
for item in v3_result["approved_plan"].unsupported:
    print("-", item)


🧠 WHAT THE AI ACTUALLY PRIORITIZED

average_net_price: 41.7%
MD_EARN_WNE_P10: 33.3%
DEBT_MDN: 25.0%

🏆 TOP 10 — ACTUAL COLLEGE DATA


,rank,college_name,matching_major_awards_2024,final_score,average_net_price,MD_EARN_WNE_P10,DEBT_MDN
0,1,Utah Valley University,4,79.82,8721.0,60162.0,8750.0
1,2,The University of Texas at Arlington,8,77.15,13755.0,72508.0,10863.0
2,3,University of California-Davis,2,74.41,17270.0,86999.0,10672.0
3,4,Florida Atlantic University,12,72.84,10225.0,61062.0,12383.0
4,5,Rhode Island College,4,70.53,10346.0,60399.0,12940.0
5,6,University of Illinois Urbana-Champaign,20,69.63,13973.0,87599.0,15446.0
6,7,The University of Texas at Dallas,35,69.32,14916.0,74762.0,13545.0
7,8,Portland State University,17,69.24,10195.0,62447.0,14167.0
8,9,Dartmouth College,17,69.06,20322.0,111162.0,12000.0
9,10,University of Wisconsin-Madison,246,68.89,14235.0,80330.0,15000.0



📊 HOW THE TOP 5 EARNED THEIR SCORES


,college_name,average_net_price_contribution,MD_EARN_WNE_P10_contribution,DEBT_MDN_contribution
0,Utah Valley University,37.47,20.49,21.85
1,The University of Texas at Arlington,30.21,27.83,19.11
2,University of California-Davis,24.08,31.08,19.25
3,Florida Atlantic University,35.72,21.11,16.00
4,Rhode Island College,35.56,20.67,14.30



⚠️ REQUIREMENTS NOT MEASURED
- Amazing Data Science program quality
- Employment outcomes specifically for Data Science
